# Boreal resilience pipeline — Taylor rolling-mean + harmonic method — all valid pixels

This notebook is the full Colab run for the boreal forest resilience project using the Taylor/Smith-style preprocessing approach: rolling-mean detrending followed by harmonic deseasoning. It avoids interpolation/gap filling and runs the event detection and benchmarking on **all valid candidate pixels**, not on stratified samples. Diagnostic maps show the valid data footprint, quality exclusions, all candidate pixels, detected events, and rolling resilience metrics.


## Colab setup

Run the next two cells first.  
They install the geospatial packages needed for raw MODIS HDF files and mount Google Drive.


In [ ]:
# Colab package install
!apt-get -qq update
!apt-get -qq install -y gdal-bin python3-gdal libgdal-dev > /dev/null
!pip -q install pyhdf xarray netCDF4 dask[complete] rasterio rioxarray pyproj bottleneck scipy statsmodels cartopy shapely

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import glob
import math
import os
import re
import shutil
import tempfile
from collections import defaultdict

import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import statsmodels.api as sm

import rasterio
import rioxarray as rxr
from pyhdf.SD import SD, SDC
from pyproj import Transformer
from scipy.stats import spearmanr, pearsonr, linregress
from scipy.optimize import curve_fit
from scipy.signal import savgol_filter, argrelmax
from osgeo import gdal

# Optional mapping support for land/ocean boundaries.
try:
    import cartopy.crs as ccrs
    import cartopy.feature as cfeature
    CARTOPY_AVAILABLE = True
except Exception as e:
    CARTOPY_AVAILABLE = False
    print("Cartopy not available; maps will fall back to plain matplotlib:", repr(e))

xr.set_options(keep_attrs=True)

## 0. Configuration
Set paths and analysis settings.

In [ ]:
print("STEP 0 - Configuration (Colab full run, Taylor rolling-mean + harmonic method)")

# ---------- General ----------
SEED = 42
np.random.seed(SEED)

# Edit this to the folder where you keep the project inside Google Drive.
PROJECT_ROOT = Path("/content/drive/MyDrive/Pythonnb")
DATA_DIR = PROJECT_ROOT / "data"
CACHE_DIR = PROJECT_ROOT / "cache_colab"
OUTPUT_DIR = PROJECT_ROOT / "outputs_resilience_taylor_harmonic_ALL_VALID_PIXELS_colab"
DIAG_DIR = OUTPUT_DIR / "diagnostics_lana_taylor_all_valid_pixels"

CACHE_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
DIAG_DIR.mkdir(parents=True, exist_ok=True)
# ---------- Recovery fit quality thresholds ----------
MIN_RECOVERY_R2 = 0.3          # minimum R² for accepting an exponential recovery fit
MIN_RECOVERY_FIT_POINTS = 6    # minimum number of finite (non-NaN) observations within the recovery window required before fitting
# ---------- Full-run controls ----------
TEST_RUN = False
MODIS_NDVI_MAX_FILES = None
TEST_MAX_TIME_STEPS = None
TEST_NOTE = (
    "Full all-valid-pixel notebook: Taylor/Smith rolling-mean detrending (5-yr window) + "
    "3rd-order harmonic deseasoning. No gap filling, no interpolation — NaNs are preserved throughout. "
    "Main benchmark uses every pixel that passes the boreal mask and data-quality filters; no stratified sampling is applied. "
    f"Recovery fit quality gates: R²>={MIN_RECOVERY_R2}, min fit points>={MIN_RECOVERY_FIT_POINTS}."
)

# ---------- Region ----------
LAT_MIN, LAT_MAX = 45.0, 75.0
LON_MIN, LON_MAX = -170.0, -50.0

# ---------- Common output grid ----------
COMMON_RES_DEG = 0.25

# ---------- Boreal mask ----------
BOREAL_MASK_FILE = PROJECT_ROOT / "MCD12C1_boreal_mask.nc"   # optional
USE_BOREAL_MASK = BOREAL_MASK_FILE.exists()

# ---------- Full candidate-pixel run ----------
# No stratified sampling is used in this notebook.
# All pixels that pass MIN_POINTS, MIN_VALID_FRACTION, MIN_VARIANCE, and the boreal mask are processed.
N_DIAGNOSTIC_LOCATIONS = 3

# ---------- Taylor rolling-mean + harmonic preprocessing ----------
# The Smith/Boers code uses rolling mean detrending followed by a third-order harmonic fit.
# For monthly data, 5 years = 60 monthly observations.
ROLLING_MEAN_YEARS = 5
OBS_PER_YEAR = 12
ROLLING_MEAN_WINDOW = ROLLING_MEAN_YEARS * OBS_PER_YEAR
HARMONIC_ORDER = 3
APPLY_EDGE_MASK = True
EDGE_MASK_MONTHS = ROLLING_MEAN_WINDOW // 2  # edge-effect region from rolling window

# ---------- Taylor-style disturbance / recovery ----------
TRANSITION_WIN_SIZE = 24          # months, before-vs-after moving-window contrast
K_VALUES = [1.5, 1.75, 2.0]       # K-multipliers for std-dev thresholding
PRIMARY_K = 1.5                   # Primary K to use for main run
TRANSITION_WIDTH_ROLL = 18        # candidate width estimate
MIN_EVENT_LENGTH = 3              # minimum candidate width / recovery fit points
MIN_SEPARATION = 12               # months between candidate transitions
RECOVERY_YEARS = 3                # Lana suggested 2-3 years rather than 5 years
RECOVERY_MONTHS = 12 * RECOVERY_YEARS
LOCAL_MIN_SEARCH_MONTHS = 8
# MIN_RECOVERY_R2 defined above (= 0.3) — do not redefine here
CLIP_ROLLING_EDGE_WINDOWS = 5     # Number of edge windows to clip from rolling metric plots

# ---------- Data quality ----------
MIN_POINTS = 24
MIN_VARIANCE = 1e-10
# Lowered from 0.70 to 0.40 to accommodate boreal MODIS NDVI pixels,
# which systematically lose ~4-5 months per year to snow masking.
# A pixel at 60°N typically has valid fraction ~0.55-0.60, so 0.70
# was excluding nearly all boreal candidates outside coastal BC.
# Recovery fit quality is enforced separately via MIN_RECOVERY_FIT_POINTS
# and MIN_RECOVERY_R2, so lowering this threshold does not compromise
# the quality of individual event fits.
MIN_VALID_FRACTION = 0.40
MIN_VALID_FRAC = MIN_VALID_FRACTION


# ---------- Event-level theoretical metrics ----------
WINDOW_SIZE = 60
WINDOW_STEP = 1
EVENT_WINDOW_MODE = "pre"   # "pre" or "center"

# ---------- Dataset paths ----------
MODIS_NDVI_FOLDER = DATA_DIR / "MOD13C2_061-20260303_192221"
VODCA_PATH = DATA_DIR / "monthly_images_VODCA_CXKu"
MODIS_GPP_PATH = DATA_DIR / "gpp_monthly_025deg_2000_2021.nc"

assert PROJECT_ROOT.exists(), f"PROJECT_ROOT not found: {PROJECT_ROOT}"
assert DATA_DIR.exists(), f"Data folder not found: {DATA_DIR}"
assert MODIS_NDVI_FOLDER.exists(), f"MODIS NDVI folder not found: {MODIS_NDVI_FOLDER}"
assert VODCA_PATH.exists(), f"VODCA monthly folder not found: {VODCA_PATH}"
assert MODIS_GPP_PATH.exists(), f"MODIS GPP monthly file not found: {MODIS_GPP_PATH}"

print("Using PROJECT_ROOT:", PROJECT_ROOT)
print("Using DATA_DIR:", DATA_DIR)
print("Using CACHE_DIR:", CACHE_DIR)
print("Using OUTPUT_DIR:", OUTPUT_DIR)
print("Diagnostics:", DIAG_DIR)
print("Region:", (LAT_MIN, LAT_MAX, LON_MIN, LON_MAX))
print("Use boreal mask:", USE_BOREAL_MASK)
print("MODIS NDVI folder:", MODIS_NDVI_FOLDER)
print("VODCA monthly folder:", VODCA_PATH)
print("MODIS GPP monthly file:", MODIS_GPP_PATH)
print("Taylor rolling mean window:", ROLLING_MEAN_WINDOW, "months")
print("Harmonic order:", HARMONIC_ORDER)
print("Edge mask months:", EDGE_MASK_MONTHS if APPLY_EDGE_MASK else 0)
print("Transition window:", TRANSITION_WIN_SIZE, "months")
print("Recovery fit window:", RECOVERY_YEARS, "years")
print("Theoretical metrics window mode:", EVENT_WINDOW_MODE)
print(TEST_NOTE)

# Global variable to pass K to transition_taylor
CURRENT_K_STD_MULTIPLIER = PRIMARY_K

## 1. Low-level helpers
Utility functions used throughout the notebook.

In [ ]:
def find_lat_lon_dims(da: xr.DataArray):
    """Find the latitude and longitude dimension names in a DataArray."""
    lat_dim = None
    lon_dim = None
    for dim in da.dims:
        d = dim.lower()
        if "lat" in d or d.startswith("y"):
            if lat_dim is None:
                lat_dim = dim
        if "lon" in d or d.startswith("x"):
            if lon_dim is None:
                lon_dim = dim
    if lat_dim is None or lon_dim is None:
        raise ValueError(f"Could not find lat/lon dims in dims={da.dims}")
    return lat_dim, lon_dim


def get_lat_lon_coords(da: xr.DataArray, lat_dim: str, lon_dim: str):
    """
    Return latitude and longitude coordinate arrays.
    Works for:
    - datasets that already have lat/lon coords
    - MODIS CMG arrays where coords are missing and must be built from indices
    """
    if lat_dim in da.coords:
        lat = da[lat_dim].values
    else:
        n = da.sizes[lat_dim]
        lat = np.linspace(LAT_MAX - 0.025, LAT_MIN + 0.025, n)

    if lon_dim in da.coords:
        lon = da[lon_dim].values
    else:
        n = da.sizes[lon_dim]
        lon = np.linspace(LON_MIN + 0.025, LON_MAX - 0.025, n)

    lat = np.asarray(lat)
    lon = np.asarray(lon)

    lat = np.squeeze(lat)
    lon = np.squeeze(lon)

    return lat, lon


def normalize_longitudes(ds: xr.Dataset | xr.DataArray):
    """
    Convert longitudes from 0..360 to -180..180 if needed.
    Keeps data sorted by longitude.
    """
    candidate_names = [c for c in list(ds.coords) + list(ds.dims) if "lon" in c.lower()]
    if not candidate_names:
        return ds

    lon_name = candidate_names[0]
    lon_vals = ds[lon_name].values

    if np.nanmax(lon_vals) > 180:
        new_lon = ((lon_vals + 180) % 360) - 180
        ds = ds.assign_coords({lon_name: new_lon}).sortby(lon_name)

    return ds


def safe_lat_lon_subset(ds_or_da, lat_name="lat", lon_name="lon",
                        lat_min=None, lat_max=None, lon_min=None, lon_max=None):
    """
    Subset a Dataset/DataArray by lat/lon using boolean masks.
    This is safer than slice(...) because it works even if coordinates are
    descending, slightly irregular, or not strictly monotonic.
    """
    out = ds_or_da

    if lat_name in out.coords and lat_min is not None and lat_max is not None:
        lat_vals = np.asarray(out[lat_name].values).squeeze()
        if lat_vals.size > 0:
            lat_lo = min(lat_min, lat_max)
            lat_hi = max(lat_min, lat_max)
            lat_mask = (out[lat_name] >= lat_lo) & (out[lat_name] <= lat_hi)
            out = out.where(lat_mask, drop=True)

    if lon_name in out.coords and lon_min is not None and lon_max is not None:
        lon_vals = np.asarray(out[lon_name].values).squeeze()
        if lon_vals.size > 0:
            lon_lo = min(lon_min, lon_max)
            lon_hi = max(lon_min, lon_max)
            lon_mask = (out[lon_name] >= lon_lo) & (out[lon_name] <= lon_hi)
            out = out.where(lon_mask, drop=True)

    return out


def spatial_subset(ds_or_da):
    """Subset to the Canada-Alaska study box."""
    probe = ds_or_da if isinstance(ds_or_da, xr.DataArray) else list(ds_or_da.data_vars.values())[0]
    lat_dim, lon_dim = find_lat_lon_dims(probe)

    if lat_dim in ds_or_da.coords and lon_dim in ds_or_da.coords:
        ds_or_da = safe_lat_lon_subset(
            ds_or_da,
            lat_name=lat_dim,
            lon_name=lon_dim,
            lat_min=LAT_MIN,
            lat_max=LAT_MAX,
            lon_min=LON_MIN,
            lon_max=LON_MAX,
        )

    return ds_or_da


def stl_deseason_detrend(da: xr.DataArray):
    """
    Deseason and detrend a monthly time series with STL.
    Returns residual, trend, seasonal.
    Residual is the working series for event detection and theoretical metrics.
    """
    da = da.dropna("time")
    if da.sizes.get("time", 0) < max(MIN_POINTS, 2 * STL_PERIOD):
        return None, None, None

    values = da.values.astype(float)
    if np.sum(np.isfinite(values)) < max(MIN_POINTS, 2 * STL_PERIOD):
        return None, None, None

    try:
        stl = STL(
            values,
            period=STL_PERIOD,
            seasonal=STL_SEASONAL,
            trend=STL_TREND,
            robust=STL_ROBUST,
        )
        res = stl.fit()
    except Exception:
        return None, None, None

    residual = xr.DataArray(
        res.resid.astype(np.float32),
        coords={"time": da.time.values},
        dims=["time"],
        name="residual"
    )
    trend = xr.DataArray(
        res.trend.astype(np.float32),
        coords={"time": da.time.values},
        dims=["time"],
        name="trend"
    )
    seasonal = xr.DataArray(
        res.seasonal.astype(np.float32),
        coords={"time": da.time.values},
        dims=["time"],
        name="seasonal"
    )
    return residual, trend, seasonal


def valid_series_fraction(da: xr.DataArray) -> float:
    if "time" not in da.dims:
        return 0.0
    return float(da.count("time") / da.sizes["time"])


def lag1_autocorrelation(values: np.ndarray) -> float:
    values = np.asarray(values, dtype=float)
    finite = np.isfinite(values)
    values = values[finite]
    if len(values) < 3:
        return np.nan
    x1 = values[:-1]
    x2 = values[1:]
    if np.std(x1) == 0 or np.std(x2) == 0:
        return np.nan
    return float(np.corrcoef(x1, x2)[0, 1])


def ar1_phi(values: np.ndarray) -> float:
    values = np.asarray(values, dtype=float)
    finite = np.isfinite(values)
    values = values[finite]
    if len(values) < 3:
        return np.nan
    x1 = values[:-1]
    x2 = values[1:]
    if np.std(x1) == 0:
        return np.nan
    phi = np.polyfit(x1, x2, 1)[0]
    return float(phi)


def ar1_lambda(values: np.ndarray) -> float:
    """
    Convert AR(1) persistence into a recovery-rate-like quantity:
    lambda = -ln(phi), valid for 0 < phi < 1
    """
    phi = ar1_phi(values)
    if np.isnan(phi):
        return np.nan
    if phi <= 0 or phi >= 1:
        return np.nan
    return float(-np.log(phi))


def glsar_a(values: np.ndarray) -> float:
    """
    GLSAR-based local stability coefficient inspired by run_fit_a_ar1 in EWS_functions.py.
    """
    x = np.asarray(values, dtype=float)
    x = x[np.isfinite(x)]
    if len(x) < 8:
        return np.nan
    if np.nanstd(x) < MIN_VARIANCE:
        return np.nan

    xw = x - np.nanmean(x)
    try:
        p0, p1 = np.polyfit(np.arange(len(xw)), xw, 1)
        xw = xw - p0 * np.arange(len(xw)) - p1

        dxw = xw[1:] - xw[:-1]
        exog = sm.add_constant(xw[:-1])
        model = sm.GLSAR(dxw, exog, rho=1)
        results = model.iterative_fit(maxiter=10)
        return float(results.params[1])
    except Exception:
        return np.nan


def safe_spearman(x, y):
    df = pd.DataFrame({"x": x, "y": y}).dropna()
    if len(df) < 3:
        return np.nan, np.nan, len(df)
    r, p = spearmanr(df["x"], df["y"])
    return float(r), float(p), len(df)


def safe_pearson(x, y):
    df = pd.DataFrame({"x": x, "y": y}).dropna()
    if len(df) < 3:
        return np.nan, np.nan, len(df)
    r, p = pearsonr(df["x"], df["y"])
    return float(r), float(p), len(df)


## 2. Data loading functions
Defines the MODIS QA and file-loading helpers.

In [ ]:
print("\nSTEP 1 - Data loading + monthly preprocessing")

import datetime as dt

def parse_modis_date_from_filename(filename: str) -> pd.Timestamp:
    m = re.search(r"\.A(\d{4})(\d{3})\.", Path(filename).name)
    if not m:
        raise ValueError(f"Could not parse MODIS date from filename: {filename}")
    year = int(m.group(1))
    doy = int(m.group(2))
    return pd.to_datetime(f"{year}-{doy}", format="%Y-%j")

def find_first_matching_sds(sd_obj, keywords):
    names = list(sd_obj.datasets().keys())
    for name in names:
        lname = name.lower()
        if all(k.lower() in lname for k in keywords):
            return name
    raise KeyError(f"No SDS matching {keywords}. Available SDS names: {names[:20]}")

def open_mod13c2_ndvi_hdf(filepath: str) -> xr.DataArray:
    """
    Read MOD13C2 monthly NDVI from HDF using pyhdf, apply QA, and return a lat/lon DataArray.
    """
    hdf = SD(filepath, SDC.READ)

    ndvi_name = find_first_matching_sds(hdf, ["monthly", "ndvi"])
    qa_name = find_first_matching_sds(hdf, ["monthly", "vi quality"])
    rel_name = find_first_matching_sds(hdf, ["monthly", "pixel reliability"])

    ndvi_raw = hdf.select(ndvi_name)[:].astype(np.int32)
    vi_quality = hdf.select(qa_name)[:].astype(np.uint16)
    pixel_reliability = hdf.select(rel_name)[:].astype(np.int16)

    valid_range_mask = (ndvi_raw >= -2000) & (ndvi_raw <= 10000)
    reliability_mask = (pixel_reliability == 0) | (pixel_reliability == 1)
    modland_qa = vi_quality & 0b11
    modland_mask = (modland_qa == 0) | (modland_qa == 1)
    usefulness = (vi_quality >> 2) & 0b1111
    usefulness_mask = usefulness <= 2

    qa_mask = valid_range_mask & reliability_mask & modland_mask & usefulness_mask
    ndvi = np.where(qa_mask, ndvi_raw.astype(np.float32) * 0.0001, np.nan)

    nlat, nlon = ndvi.shape
    lat = 90 - 0.05 * (np.arange(nlat) + 0.5)
    lon = -180 + 0.05 * (np.arange(nlon) + 0.5)

    da = xr.DataArray(
        ndvi,
        dims=("lat", "lon"),
        coords={"lat": lat.astype(np.float32), "lon": lon.astype(np.float32)},
        name="NDVI"
    )

    da = normalize_longitudes(da)
    da = safe_lat_lon_subset(da, "lat", "lon", LAT_MIN, LAT_MAX, LON_MIN, LON_MAX)

    ts = parse_modis_date_from_filename(filepath)
    da = da.expand_dims(time=[pd.Timestamp(ts.year, ts.month, 1)])
    return da

def standardize_time_dimension(ds: xr.Dataset) -> xr.Dataset:
    if "time" in ds.dims or "time" in ds.coords:
        return ds
    candidate_names = []
    for name in list(ds.dims) + list(ds.coords):
        lname = name.lower()
        if "time" in lname or "date" in lname or lname in ["month", "t"]:
            candidate_names.append(name)
    if candidate_names:
        ds = ds.rename({candidate_names[0]: "time"})
    return ds

def pick_preferred_data_var(ds: xr.Dataset, preferred_names: list[str], dataset_name: str) -> str:
    lower_map = {name.lower(): name for name in ds.data_vars}
    for cand in preferred_names:
        if cand.lower() in lower_map:
            return lower_map[cand.lower()]
    if len(ds.data_vars) == 0:
        raise ValueError(f"{dataset_name}: no data variables found")
    chosen = list(ds.data_vars)[0]
    print(f"{dataset_name}: preferred variable not found; using first variable: {chosen}")
    return chosen

def to_month_start_datetimeindex(time_values) -> pd.DatetimeIndex:
    out = []
    for t in list(time_values):
        if isinstance(t, pd.Timestamp):
            ts = t
        elif isinstance(t, np.datetime64):
            ts = pd.Timestamp(t)
        elif isinstance(t, dt.datetime):
            ts = pd.Timestamp(t)
        elif hasattr(t, "year") and hasattr(t, "month"):
            ts = pd.Timestamp(int(t.year), int(t.month), 1)
        else:
            ts = pd.Timestamp(str(t))
        out.append(pd.Timestamp(ts.year, ts.month, 1))
    return pd.DatetimeIndex(out)

def standardize_monthly_time_coord(da: xr.DataArray) -> xr.DataArray:
    if "time" not in da.coords:
        raise ValueError("DataArray has no time coordinate.")
    da = da.assign_coords(time=to_month_start_datetimeindex(da["time"].values))
    da = da.sortby("time")
    if da.indexes["time"].duplicated().any():
        da = da.groupby("time").mean()
    return da

def load_vodca_monthly(vodca_path: Path) -> xr.DataArray:
    cache_path = CACHE_DIR / ("vodca_monthly_test.nc" if TEST_RUN else "vodca_monthly_full.nc")
    if cache_path.exists():
        print(f"VODCA: using cached monthly file {cache_path.name}")
        ds = xr.open_dataset(cache_path)
        return ds["VODCA"]

    print(f"Reading VODCA monthly files from: {vodca_path}")
    files = sorted(vodca_path.glob("*.nc"))
    if len(files) == 0:
        raise ValueError(f"No NetCDF files found in {vodca_path}")

    print(f"Found {len(files)} monthly files")
    parts = []
    for i, fp in enumerate(files, start=1):
        if i == 1 or i % 24 == 0 or i == len(files):
            print(f"  VODCA file {i}/{len(files)}: {fp.name}")
        ds_single = xr.open_dataset(fp, engine="netcdf4")
        try:
            try:
                ds_single = xr.decode_cf(ds_single)
            except Exception:
                pass
            ds_single = standardize_time_dimension(ds_single)
            ds_single = normalize_longitudes(ds_single)
            ds_single = spatial_subset(ds_single)

            vod_var = pick_preferred_data_var(
                ds_single,
                ["VODCA_CXKu", "VODCA_CXKU", "VI", "vod", "vodca"],
                "VODCA monthly"
            )
            da_single = ds_single[vod_var].astype(np.float32)
            rename_dict = {}
            for dim in da_single.dims:
                if dim.lower() in ["latitude", "y"]:
                    rename_dict[dim] = "lat"
                elif dim.lower() in ["longitude", "x"]:
                    rename_dict[dim] = "lon"
            if rename_dict:
                da_single = da_single.rename(rename_dict)
            da_single = da_single.load()
            parts.append(da_single)
        finally:
            ds_single.close()

    da = xr.concat(parts, dim="time")
    da = standardize_monthly_time_coord(da)

    if TEST_MAX_TIME_STEPS is not None and da.sizes.get("time", 0) > TEST_MAX_TIME_STEPS:
        da = da.isel(time=slice(0, TEST_MAX_TIME_STEPS))

    da.name = "VODCA"
    da.to_dataset(name="VODCA").to_netcdf(cache_path)
    print(f"VODCA: wrote cache {cache_path.name}")
    return da

def load_modis_ndvi_monthly() -> xr.DataArray:
    cache_path = CACHE_DIR / ("modis_ndvi_monthly_test.nc" if TEST_RUN else "modis_ndvi_monthly_full.nc")
    if cache_path.exists():
        print(f"MODIS NDVI: using cached monthly file {cache_path.name}")
        ds = xr.open_dataset(cache_path)
        return ds["NDVI"]

    hdf_files = sorted(glob.glob(str(MODIS_NDVI_FOLDER / "*.hdf")))
    if len(hdf_files) == 0:
        raise FileNotFoundError(f"No MOD13C2 HDF files found in {MODIS_NDVI_FOLDER}")

    if MODIS_NDVI_MAX_FILES is not None:
        hdf_files = hdf_files[:MODIS_NDVI_MAX_FILES]

    print(f"MODIS NDVI: reading {len(hdf_files)} monthly HDF files")
    parts = []
    for i, fp in enumerate(hdf_files, start=1):
        if i == 1 or i % 12 == 0 or i == len(hdf_files):
            print(f"  MODIS NDVI file {i}/{len(hdf_files)}: {Path(fp).name}")
        parts.append(open_mod13c2_ndvi_hdf(fp))

    da = xr.concat(parts, dim="time").sortby("time")
    da = standardize_monthly_time_coord(da)
    da.name = "NDVI"
    da.to_dataset(name="NDVI").to_netcdf(cache_path)
    print(f"MODIS NDVI: wrote cache {cache_path.name}")
    return da

def load_mod17_gpp_monthly_preprocessed(gpp_path: Path) -> xr.DataArray:
    cache_path = CACHE_DIR / ("gpp_monthly_preprocessed_test.nc" if TEST_RUN else "gpp_monthly_preprocessed_full.nc")
    if cache_path.exists():
        print(f"MOD17 GPP: using cached monthly file {cache_path.name}")
        ds = xr.open_dataset(cache_path)
        return ds["GPP"]

    print(f"Reading preprocessed GPP file from: {gpp_path}")
    ds = xr.open_dataset(gpp_path, engine="netcdf4")
    try:
        possible_vars = list(ds.data_vars)
        preferred_names = ["GPP", "gpp", "Gpp_500m"]
        gpp_var = None
        for name in preferred_names:
            for v in possible_vars:
                if v == name or name.lower() in v.lower():
                    gpp_var = v
                    break
            if gpp_var is not None:
                break
        if gpp_var is None:
            gpp_var = possible_vars[0]

        da = ds[gpp_var].astype(np.float32)

        rename_dict = {}
        for dim in da.dims:
            if dim.lower() in ["latitude", "y"]:
                rename_dict[dim] = "lat"
            elif dim.lower() in ["longitude", "x"]:
                rename_dict[dim] = "lon"
        if rename_dict:
            da = da.rename(rename_dict)

        da = normalize_longitudes(da)
        da = safe_lat_lon_subset(da, "lat", "lon", LAT_MIN, LAT_MAX, LON_MIN, LON_MAX)
        da = standardize_monthly_time_coord(da)

        if TEST_MAX_TIME_STEPS is not None and da.sizes.get("time", 0) > TEST_MAX_TIME_STEPS:
            da = da.isel(time=slice(0, TEST_MAX_TIME_STEPS))

        da.name = "GPP"
        da.to_dataset(name="GPP").to_netcdf(cache_path)
        print(f"MOD17 GPP: wrote cache {cache_path.name}")
        return da
    finally:
        ds.close()


## 2b. Load datasets (full run)
Loads the three datasets, aligns them to the VODCA grid, and intersects the full monthly time range.


In [ ]:
import pandas as pd
import xarray as xr

print("Loading VODCA monthly from pre-aggregated monthly files...")
vodca_monthly = load_vodca_monthly(VODCA_PATH)
print("VODCA monthly shape:", dict(vodca_monthly.sizes))
print("VODCA time range:", str(vodca_monthly.time.values[0]), "to", str(vodca_monthly.time.values[-1]))

target_lat = vodca_monthly["lat"]
target_lon = vodca_monthly["lon"]

def align_to_target_grid(da: xr.DataArray, target_lat: xr.DataArray, target_lon: xr.DataArray) -> xr.DataArray:
    lat_dim, lon_dim = find_lat_lon_dims(da)
    da = normalize_longitudes(da)
    if lat_dim != "lat" or lon_dim != "lon":
        da = da.rename({lat_dim: "lat", lon_dim: "lon"})
    da = da.sortby("lat").sortby("lon")
    return da.interp(lat=target_lat, lon=target_lon, method="nearest")

print("Loading MODIS NDVI monthly...")
modis_ndvi = load_modis_ndvi_monthly()
modis_ndvi = align_to_target_grid(modis_ndvi, target_lat, target_lon)
modis_ndvi = standardize_monthly_time_coord(modis_ndvi)
print("MODIS NDVI aligned shape:", dict(modis_ndvi.sizes))

print("Loading preprocessed MOD17 GPP monthly 0.25° file...")
gpp_monthly = load_mod17_gpp_monthly_preprocessed(MODIS_GPP_PATH)
gpp_monthly = align_to_target_grid(gpp_monthly, target_lat, target_lon)
gpp_monthly = standardize_monthly_time_coord(gpp_monthly)
print("MOD17 GPP aligned shape:", dict(gpp_monthly.sizes))

vodca_monthly = standardize_monthly_time_coord(vodca_monthly)

common_time = modis_ndvi.indexes["time"].intersection(vodca_monthly.indexes["time"]).intersection(gpp_monthly.indexes["time"])
common_time = pd.DatetimeIndex(common_time).sort_values()

print("Common monthly steps across all three datasets:", len(common_time))
print("Common time start/end:", common_time[0], common_time[-1])

modis_ndvi = modis_ndvi.sel(time=common_time)
vodca_monthly = vodca_monthly.sel(time=common_time)
gpp_monthly = gpp_monthly.sel(time=common_time)

ds_vodca = vodca_monthly.to_dataset(name="VODCA")
ds_gpp = gpp_monthly.to_dataset(name="GPP")
vodca_var = "VODCA"
gpp_var = "GPP"

print("Final dataset shapes on common grid:")
print("  MODIS NDVI:", dict(modis_ndvi.sizes))
print("  VODCA:", dict(ds_vodca[vodca_var].sizes))
print("  GPP:", dict(ds_gpp[gpp_var].sizes))


## 3. Boreal mask
Loads and aligns the boreal forest mask.

In [ ]:
print("\nSTEP 2 - Loading boreal forest mask")

def load_boreal_mask_for_grid(target_da: xr.DataArray):
    if not USE_BOREAL_MASK:
        print("No boreal mask file supplied or file not found. Using study-area bounding box only.")
        return None

    print("Loading boreal mask from:", BOREAL_MASK_FILE)
    ds_mask = xr.open_dataset(BOREAL_MASK_FILE)
    boreal_mask = ds_mask["boreal_mask"]

    mask_lat_dim, mask_lon_dim = find_lat_lon_dims(boreal_mask)
    if mask_lat_dim != "lat" or mask_lon_dim != "lon":
        boreal_mask = boreal_mask.rename({mask_lat_dim: "lat", mask_lon_dim: "lon"})

    boreal_mask = boreal_mask.sortby("lat").sortby("lon")

    lat_dim, lon_dim = find_lat_lon_dims(target_da)
    target_lat, target_lon = get_lat_lon_coords(target_da, lat_dim, lon_dim)

    target_lat = np.asarray(target_lat, dtype=float).ravel()
    target_lon = np.asarray(target_lon, dtype=float).ravel()
    target_lat = target_lat[~np.isnan(target_lat)]
    target_lon = target_lon[~np.isnan(target_lon)]

    if target_lat.size == 0 or target_lon.size == 0:
        raise ValueError("Target grid has empty lat/lon coordinates during boreal-mask alignment.")

    boreal_mask_aligned = boreal_mask.interp(
        lat=target_lat,
        lon=target_lon,
        method="nearest"
    )

    rename_dict = {}
    if "lat" in boreal_mask_aligned.dims and lat_dim not in boreal_mask_aligned.dims:
        rename_dict["lat"] = lat_dim
    if "lon" in boreal_mask_aligned.dims and lon_dim not in boreal_mask_aligned.dims:
        rename_dict["lon"] = lon_dim
    if rename_dict:
        boreal_mask_aligned = boreal_mask_aligned.rename(rename_dict)

    boreal_mask_aligned = boreal_mask_aligned.astype(bool)
    print("Boreal mask successfully aligned to target grid.")
    return boreal_mask_aligned

boreal_mask_on_grid = load_boreal_mask_for_grid(modis_ndvi)


In [ ]:
boreal_mask_on_grid = load_boreal_mask_for_grid(modis_ndvi)


## 5. Lana diagnostics for Taylor preprocessing

In [ ]:
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature

print("\nSTEP 2B - Visualizing boreal forest mask")

def plot_boreal_mask_on_map(data_da, boreal_mask_on_grid):

    if boreal_mask_on_grid is None:
        print("No boreal mask available to plot.")
        return

    mask_plot = boreal_mask_on_grid.astype(float)

    fig = plt.figure(figsize=(10, 6))
    ax = plt.axes(projection=ccrs.PlateCarree())

    # Plot boreal mask
    im = ax.pcolormesh(
        data_da["lon"],
        data_da["lat"],
        mask_plot,
        cmap="Greens",
        shading="auto",
        vmin=0,
        vmax=1
    )

    # Add land / ocean / boundaries
    ax.add_feature(cfeature.OCEAN, facecolor="#d9eef7")
    ax.add_feature(cfeature.LAND, facecolor="#f0f0e8", edgecolor="black", linewidth=0.4)
    ax.add_feature(cfeature.COASTLINE, linewidth=0.8)
    ax.add_feature(cfeature.BORDERS, linestyle=":", linewidth=0.5)

    # Focus on boreal region (adjust if needed)
    ax.set_extent([LON_MIN, LON_MAX, LAT_MIN, LAT_MAX], crs=ccrs.PlateCarree())

    # Colorbar
    cbar = plt.colorbar(im, ax=ax, shrink=0.7)
    cbar.set_label("Boreal mask (1 = boreal)")

    plt.title("Boreal forest mask (aligned to analysis grid)")
    plt.tight_layout()
    plt.show()

# Call it
plot_boreal_mask_on_map(modis_ndvi, boreal_mask_on_grid)

print("Mask shape:", boreal_mask_on_grid.shape)
print("Data shape:", modis_ndvi.isel(time=0).shape)


## 4. Candidate-pixel functions
Build all valid candidate pixels. No stratified sampling is used in the main analysis.


In [ ]:
print("\nSTEP 3 - Candidate pixel selection for all-valid-pixel run")

def build_candidate_pixel_table(da: xr.DataArray, dataset_name: str):
    """
    Fast vectorized candidate-pixel builder.
    A pixel is a candidate if it has enough valid time points,
    enough valid fraction, enough variance, and (optionally) falls
    inside the boreal mask.

    This table is used directly for the main all-valid-pixel run.
    There is no stratified sampling step in this notebook.
    """
    lat_dim, lon_dim = find_lat_lon_dims(da)

    empty_cols = ["dataset", "pixel_id", "lat_idx", "lon_idx", "lat", "lon", "nan_fraction", "valid_count", "valid_fraction", "variance"]

    if da.sizes.get(lat_dim, 0) == 0:
        print(f"{dataset_name}: latitude dimension is empty. Returning an empty candidate table.")
        return pd.DataFrame(columns=empty_cols)
    if da.sizes.get(lon_dim, 0) == 0:
        print(f"{dataset_name}: longitude dimension is empty. Returning an empty candidate table.")
        return pd.DataFrame(columns=empty_cols)

    # Force a consistent dimension order before building 2D masks.
    da = da.transpose("time", lat_dim, lon_dim)
    print(f"{dataset_name}: dims after transpose = {da.dims}")
    print(f"{dataset_name}: shape after transpose = {da.shape}")

    lat_values, lon_values = get_lat_lon_coords(da, lat_dim, lon_dim)
    if np.asarray(lat_values).size == 0 or np.asarray(lon_values).size == 0:
        print(f"{dataset_name}: coordinate arrays are empty after extraction. Returning an empty candidate table.")
        return pd.DataFrame(columns=empty_cols)

    print(f"{dataset_name}: loading boreal mask...")
    boreal_mask = load_boreal_mask_for_grid(da)

    print(f"{dataset_name}: computing valid-count / valid-fraction / variance masks...")
    valid_count = da.count("time")
    valid_fraction = valid_count / da.sizes["time"]
    variance = da.var("time", skipna=True)

    candidate_mask = (
        (valid_count >= MIN_POINTS)
        & (valid_fraction >= MIN_VALID_FRACTION)
        & (variance >= MIN_VARIANCE)
    )

    if boreal_mask is not None:
        candidate_mask = candidate_mask & boreal_mask.astype(bool)

    print(f"{dataset_name}: materializing candidate mask into memory...")
    if hasattr(candidate_mask.data, "compute"):
        candidate_mask = candidate_mask.compute()
        valid_fraction = valid_fraction.compute()
        valid_count = valid_count.compute()
        variance = variance.compute()
    else:
        candidate_mask = candidate_mask.load()
        valid_fraction = valid_fraction.load()
        valid_count = valid_count.load()
        variance = variance.load()

    candidate_values = np.asarray(candidate_mask.values, dtype=bool)
    if candidate_values.ndim != 2:
        raise ValueError(f"{dataset_name}: candidate mask is not 2D. Shape={candidate_values.shape}")

    ij = np.argwhere(candidate_values)
    print(f"{dataset_name}: candidate pixels found = {len(ij)}")

    if len(ij) == 0:
        return pd.DataFrame(columns=empty_cols)

    rows = pd.DataFrame({
        "dataset": dataset_name,
        "pixel_id": np.arange(len(ij), dtype=int),
        "lat_idx": ij[:, 0].astype(int),
        "lon_idx": ij[:, 1].astype(int),
    })
    rows["lat"] = lat_values[rows["lat_idx"].to_numpy()]
    rows["lon"] = lon_values[rows["lon_idx"].to_numpy()]

    vc_values = np.asarray(valid_count.values, dtype=float)
    vf_values = np.asarray(valid_fraction.values, dtype=float)
    var_values = np.asarray(variance.values, dtype=float)

    ii = rows["lat_idx"].to_numpy()
    jj = rows["lon_idx"].to_numpy()
    rows["valid_count"] = vc_values[ii, jj]
    rows["valid_fraction"] = vf_values[ii, jj]
    rows["nan_fraction"] = 1.0 - rows["valid_fraction"]
    rows["variance"] = var_values[ii, jj]

    # Keep this column name for compatibility with downstream code and older outputs.
    rows["sampled_pixel_id"] = rows["pixel_id"]

    return rows


def use_all_valid_pixels(candidate_df: pd.DataFrame, dataset_name: str) -> pd.DataFrame:
    """
    Return all candidate pixels for the main run.
    This replaces stratified sampling.
    """
    df = candidate_df.copy().reset_index(drop=True)
    if len(df) == 0:
        print(f"{dataset_name}: no valid candidates to process.")
        return df
    df["pixel_id"] = np.arange(len(df), dtype=int)
    df["sampled_pixel_id"] = df["pixel_id"]
    print(f"{dataset_name}: using all {len(df)} valid candidate pixels for event detection and benchmarking.")
    return df


## 4b. Build all candidate-pixel tables
This cell builds candidate tables and passes all valid pixels into the main run. The old stratified-sampling step has been removed.


In [ ]:
print("Loaded dataset summary:")
print("  MODIS:", dict(modis_ndvi.sizes))
print("  VODCA:", dict(ds_vodca[vodca_var].sizes))
print("  GPP:", dict(ds_gpp[gpp_var].sizes))


In [ ]:
print("\nSTEP 3b — Build candidate tables and use all valid pixels")

print("Building candidate pixel tables...")
modis_candidates = build_candidate_pixel_table(modis_ndvi, "MODIS_NDVI")
vodca_candidates = build_candidate_pixel_table(ds_vodca[vodca_var], "VODCA")
gpp_candidates   = build_candidate_pixel_table(ds_gpp[gpp_var], "GPP")

print(f"Candidate counts — MODIS: {len(modis_candidates)}, VODCA: {len(vodca_candidates)}, GPP: {len(gpp_candidates)}")

print("\nUsing all valid candidate pixels. No stratified sampling is applied.")
modis_sampled_pixels = use_all_valid_pixels(modis_candidates, "MODIS_NDVI")
vodca_sampled_pixels = use_all_valid_pixels(vodca_candidates, "VODCA")
gpp_sampled_pixels   = use_all_valid_pixels(gpp_candidates, "GPP")

# Save with both legacy and explicit filenames for clarity.
modis_sampled_pixels.to_csv(OUTPUT_DIR / "modis_all_valid_pixels.csv", index=False)
vodca_sampled_pixels.to_csv(OUTPUT_DIR / "vodca_all_valid_pixels.csv", index=False)
gpp_sampled_pixels.to_csv(OUTPUT_DIR / "gpp_all_valid_pixels.csv", index=False)
modis_sampled_pixels.to_csv(OUTPUT_DIR / "modis_sampled_pixels_LEGACY_NAME_all_valid.csv", index=False)
vodca_sampled_pixels.to_csv(OUTPUT_DIR / "vodca_sampled_pixels_LEGACY_NAME_all_valid.csv", index=False)
gpp_sampled_pixels.to_csv(OUTPUT_DIR / "gpp_sampled_pixels_LEGACY_NAME_all_valid.csv", index=False)

print("\nMAIN all-valid-pixel counts:")
print("  MODIS:", len(modis_sampled_pixels))
print("  VODCA:", len(vodca_sampled_pixels))
print("  GPP:", len(gpp_sampled_pixels))

# ---------- DIAGNOSTICS ONLY: common-location candidates ----------
# These are NOT used in the main event detection or benchmark.
# They are only used to make visually fair plots at the same lat/lon locations.

def _vf_2d_for_diag(da: xr.DataArray):
    lat_dim, lon_dim = find_lat_lon_dims(da)
    da2 = da.transpose("time", lat_dim, lon_dim)
    return np.isfinite(da2).mean(dim="time")


def _vc_2d_for_diag(da: xr.DataArray):
    lat_dim, lon_dim = find_lat_lon_dims(da)
    da2 = da.transpose("time", lat_dim, lon_dim)
    return np.isfinite(da2).sum(dim="time")


def _var_2d_for_diag(da: xr.DataArray):
    lat_dim, lon_dim = find_lat_lon_dims(da)
    da2 = da.transpose("time", lat_dim, lon_dim)
    return da2.var(dim="time", skipna=True)


def build_common_diagnostic_pixel_table(modis_da, vodca_da, gpp_da):
    """
    Builds a relaxed common-location table ONLY for diagnostics.
    It does not control the main analysis.
    """
    modis_vf = _vf_2d_for_diag(modis_da)
    vodca_vf = _vf_2d_for_diag(vodca_da)
    gpp_vf = _vf_2d_for_diag(gpp_da)

    modis_vc = _vc_2d_for_diag(modis_da)
    vodca_vc = _vc_2d_for_diag(vodca_da)
    gpp_vc = _vc_2d_for_diag(gpp_da)

    modis_var2 = _var_2d_for_diag(modis_da)
    vodca_var2 = _var_2d_for_diag(vodca_da)
    gpp_var2 = _var_2d_for_diag(gpp_da)

    # For diagnostics we are less strict than the main sampler, because we only need
    # a few shared locations to visualize raw/detrended behavior.
    diag_min_valid = min(MIN_VALID_FRACTION, 0.25)
    diag_min_points = min(MIN_POINTS, 60)

    mask = (
        (modis_vf >= diag_min_valid) & (vodca_vf >= diag_min_valid) & (gpp_vf >= diag_min_valid) &
        (modis_vc >= diag_min_points) & (vodca_vc >= diag_min_points) & (gpp_vc >= diag_min_points) &
        (modis_var2 > 0) & (vodca_var2 > 0) & (gpp_var2 > 0)
    )

    if boreal_mask_on_grid is not None:
        mask = mask & boreal_mask_on_grid.astype(bool)

    if hasattr(mask.data, "compute"):
        mask = mask.compute()
    else:
        mask = mask.load()

    yy, xx = np.where(np.asarray(mask.values, dtype=bool))
    lat_vals = modis_da["lat"].values
    lon_vals = modis_da["lon"].values

    rows = []
    for k, (iy, ix) in enumerate(zip(yy, xx)):
        rows.append({
            "dataset": "COMMON_DIAGNOSTIC_ONLY",
            "pixel_id": k,
            "lat_idx": int(iy),
            "lon_idx": int(ix),
            "lat": float(lat_vals[iy]),
            "lon": float(lon_vals[ix]),
            "modis_valid_fraction": float(modis_vf.values[iy, ix]),
            "vodca_valid_fraction": float(vodca_vf.values[iy, ix]),
            "gpp_valid_fraction": float(gpp_vf.values[iy, ix]),
        })
    return pd.DataFrame(rows)


def sample_common_diagnostic_pixels(candidate_df: pd.DataFrame, n: int = N_DIAGNOSTIC_LOCATIONS, seed: int = SEED):
    if len(candidate_df) == 0:
        return candidate_df.copy()
    n = min(n, len(candidate_df))
    return candidate_df.sample(n=n, random_state=seed).reset_index(drop=True)

common_diagnostic_candidates = build_common_diagnostic_pixel_table(modis_ndvi, ds_vodca[vodca_var], ds_gpp[gpp_var])
common_diagnostic_pixels = sample_common_diagnostic_pixels(common_diagnostic_candidates)

common_diagnostic_candidates.to_csv(OUTPUT_DIR / "common_diagnostic_candidate_pixels.csv", index=False)
common_diagnostic_pixels.to_csv(OUTPUT_DIR / "common_diagnostic_pixels.csv", index=False)

print("\nDIAGNOSTIC common-location pixels only:")
print("  candidates:", len(common_diagnostic_candidates))
print("  selected:", len(common_diagnostic_pixels))
if len(common_diagnostic_pixels) > 0:
    display(common_diagnostic_pixels.head())
else:
    print("  No common diagnostic pixels found. Diagnostic plots will fall back to dataset-specific sampled pixels.")

## 5. Disturbance and recovery functions
Build pixel-level anomaly and recovery records.

In [ ]:
print("\nSTEP 4 - Disturbance and recovery extraction (Taylor rolling-mean + harmonic method)")

TAYLOR_P0 = [-0.05, -0.1]
TAYLOR_BOUNDS = ([-np.inf, -np.inf], [0.0, 0.0])  # negative residual recovering toward zero

def exp_fit(x, a, b):
    return a * np.exp(b * x)

def exp_jac(x, a, b):
    return np.array([np.exp(b * x), a * x * np.exp(b * x)]).T

def get_rsq(obs, pred):
    obs = np.asarray(obs, dtype=float)
    pred = np.asarray(pred, dtype=float)
    finite = np.isfinite(obs) & np.isfinite(pred)
    if finite.sum() < 3:
        return np.nan
    obs = obs[finite]
    pred = pred[finite]
    ss_res = np.nansum((obs - pred) ** 2)
    ss_tot = np.nansum((obs - np.nanmean(obs)) ** 2)
    if ss_tot == 0:
        return np.nan
    return float(1 - ss_res / ss_tot)

# ---------- Taylor / Smith rolling mean + harmonic preprocessing ----------

def runmean_residual(values: np.ndarray, window: int) -> np.ndarray:
    """
    Rolling-mean detrender matching the Smith/Boers script logic:
    returns x - rolling_mean(x), using nanmean and edge windows.
    No interpolation or gap filling is performed.
    """
    x = np.asarray(values, dtype=float)
    n = len(x)
    xs = np.full_like(x, np.nan, dtype=float)
    half = window // 2
    if n == 0:
        return xs
    for i in range(n):
        lo = max(0, i - half)
        hi = min(n, i + half + 1)
        xs[i] = np.nanmean(x[lo:hi])
    return x - xs


def harmonic_fit_residual(series: pd.Series, order: int = 3) -> pd.Series:
    """
    Third-order harmonic removal using statsmodels OLS with missing='drop'.
    This does not fill gaps: original NaNs remain NaN in the residual.
    """
    ser = series.copy()
    x_dates = pd.DatetimeIndex(ser.index)
    y = ser.values.astype(float)

    x_years = np.array([(d - pd.Timestamp("1970-01-01")).days for d in x_dates]) / 365.25
    x_rad = x_years * 2 * np.pi

    nr_indep = order * 2 + 2
    indep = np.empty((len(y), nr_indep), dtype=float)
    indep[:, 0] = 1.0
    indep[:, 1] = x_rad
    col = 2
    for freq in range(1, order + 1):
        indep[:, col] = np.cos(x_rad * freq); col += 1
        indep[:, col] = np.sin(x_rad * freq); col += 1

    finite = np.isfinite(y)
    if finite.sum() < nr_indep + 3:
        return pd.Series(np.full_like(y, np.nan, dtype=float), index=ser.index)

    model = sm.OLS(y, indep, missing="drop").fit()
    fitted = indep @ model.params
    resid = y - fitted
    resid[~finite] = np.nan
    return pd.Series(resid, index=ser.index)


def taylor_deseason_detrend(da_1d: xr.DataArray):
    """
    Taylor/Smith preprocessing for monthly data:
    1. rolling-mean detrending using a 5-year window
    2. third-order harmonic deseasoning/detrending on the rolling residual
    3. optional edge masking because rolling windows are unreliable at the ends

    No interpolation and no da.dropna('time') are used.
    """
    times = pd.to_datetime([pd.Timestamp(str(t)) for t in da_1d.time.values])
    raw_values = da_1d.values.astype(float)
    raw_ser = pd.Series(raw_values, index=pd.DatetimeIndex(times))

    if np.isfinite(raw_values).sum() < MIN_POINTS:
        return None, None, None

    rolling_resid_values = runmean_residual(raw_values, ROLLING_MEAN_WINDOW)
    rolling_resid_ser = pd.Series(rolling_resid_values, index=raw_ser.index)
    residual_ser = harmonic_fit_residual(rolling_resid_ser, order=HARMONIC_ORDER)

    if APPLY_EDGE_MASK and len(residual_ser) > 2 * EDGE_MASK_MONTHS:
        residual_ser.iloc[:EDGE_MASK_MONTHS] = np.nan
        residual_ser.iloc[-EDGE_MASK_MONTHS:] = np.nan

    # Approximate trend+seasonal removed component for diagnostics.
    residual_values = residual_ser.values.astype(float)
    removed = raw_values - residual_values
    removed[~np.isfinite(raw_values)] = np.nan

    residual = xr.DataArray(residual_values, coords={"time": da_1d.time.values}, dims=["time"], name="taylor_residual")
    removed_da = xr.DataArray(removed, coords={"time": da_1d.time.values}, dims=["time"], name="removed_trend_season")
    seasonal_da = xr.DataArray(np.full_like(raw_values, np.nan, dtype=float), coords={"time": da_1d.time.values}, dims=["time"], name="harmonic_component_not_stored")
    return residual, removed_da, seasonal_da

# ---------- Transition detection and recovery fitting ----------

def enforce_min_separation_times(event_times: list[int], min_sep: int) -> list[int]:
    if len(event_times) == 0:
        return []
    kept = [int(event_times[0])]
    for t in event_times[1:]:
        if int(t) - kept[-1] >= min_sep:
            kept.append(int(t))
    return kept


def _smooth_with_nan_support(values: np.ndarray, window: int = 7, polyorder: int = 1) -> np.ndarray:
    """
    Smooths only for transition-strength estimation. This is not gap-filling the final series;
    it is just a temporary helper to make peak detection less noisy.
    """
    arr = np.asarray(values, dtype=float).copy()
    finite = np.isfinite(arr)
    if finite.sum() < max(window, polyorder + 2):
        return arr
    idx = np.arange(len(arr))
    arr[~finite] = np.interp(idx[~finite], idx[finite], arr[finite])
    if window % 2 == 0:
        window += 1
    window = min(window, len(arr) if len(arr) % 2 == 1 else len(arr) - 1)
    if window < polyorder + 2 or window < 3:
        return arr
    return savgol_filter(arr, window_length=window, polyorder=polyorder, deriv=0)


def transition_taylor(x, win_size, k_std_multiplier: float):
    """
    Taylor-style transition detector:
    moving-window before-vs-after contrast, smoothing, standard deviation thresholding,
    local maxima, and rough width estimate.
    """
    x = np.asarray(x, dtype=float)
    av_derivative = np.full(x.shape, np.nan, dtype=float)
    half_window = int(win_size / 2)
    ln = x.shape[0]

    for i in range(half_window, ln - half_window):
        before = x[i - half_window:i]
        after = x[i:i + half_window]
        if np.isfinite(before).sum() >= max(3, half_window // 3) and np.isfinite(after).sum() >= max(3, half_window // 3):
            av_derivative[i] = np.nanmean(before) - np.nanmean(after)

    av_derivative = _smooth_with_nan_support(av_derivative, window=7, polyorder=1)

    finite = np.isfinite(av_derivative)
    if finite.sum() == 0:
        return np.array([], dtype=int), []

    # Use K-multiplier * standard deviation for thresholding instead of percentile
    std_dev = np.nanstd(av_derivative[finite])
    thr = k_std_multiplier * std_dev # disturbances are positive changes in av_derivative

    av_derivative_masked = av_derivative.copy()
    av_derivative_masked[av_derivative < thr] = 0

    transition_times = argrelmax(np.nan_to_num(av_derivative_masked, nan=0.0), order=1)[0]

    av = av_derivative_masked.copy()
    av[av > 0] = 1
    av[np.isnan(av)] = 0
    cs = pd.Series(av).rolling(TRANSITION_WIDTH_ROLL, min_periods=1).sum()

    widths = []
    half_roll = TRANSITION_WIDTH_ROLL // 2
    for t in transition_times:
        lo = max(0, t - half_roll)
        hi = min(len(cs), t + half_roll)
        width = np.nanmax(cs.values[lo:hi])
        widths.append(float(width) if np.isfinite(width) else np.nan)

    return transition_times.astype(int), widths


def fit_transition_taylor(transition_idx: int, detrended_resid: pd.Series, raw: pd.Series, width_months: int | None = None, skip_quality_gates: bool = False):
    """
    Empirical recovery fitting on Taylor residuals.
    Fits a negative exponential recovery from the local trough toward zero.
    """
    transition_date = pd.Timestamp(detrended_resid.index[transition_idx])
    end_date = transition_date + pd.DateOffset(months=RECOVERY_MONTHS)
    prev_date = transition_date - pd.DateOffset(months=RECOVERY_MONTHS)

    if end_date > pd.Timestamp(detrended_resid.index.max()):
        return None

    fitting = detrended_resid[(detrended_resid.index >= transition_date) & (detrended_resid.index <= end_date)].copy()
    fitting_raw = raw[(raw.index >= transition_date) & (raw.index <= end_date)].copy()

    if np.isfinite(fitting.values).sum() < MIN_EVENT_LENGTH:
        return None

    look_values = fitting.values[:min(LOCAL_MIN_SEARCH_MONTHS, len(fitting))].astype(float)
    if np.isfinite(look_values).sum() < MIN_EVENT_LENGTH:
        return None
    try:
        armin = int(np.nanargmin(look_values))
    except Exception:
        return None

    fitting_min = fitting.iloc[armin:].copy()
    fitting_raw_min = fitting_raw.iloc[armin:].copy()
    y_all = fitting_min.values.astype(float)
    finite = np.isfinite(y_all)
    if finite.sum() < MIN_EVENT_LENGTH:
        return None

    # The Taylor recovery example fits negative residuals after a trough.
    # Use finite points only, but keep dates for plotting.
    x_all = np.arange(len(y_all), dtype=float)
    x = x_all[finite]
    y = y_all[finite]

    if np.nanmin(y) >= 0:
        return None

    raw_drop = float(fitting_raw.iloc[0] - fitting_raw.iloc[armin]) if len(fitting_raw) > armin else np.nan

    rsq = np.nan # Initialize rsq
    fit_a = np.nan # Initialize fit_a
    fit_b = np.nan # Initialize fit_b
    lambda_emp = np.nan # Initialize lambda_emp

    try:
        p0 = [min(float(y[0]), -1e-4), -0.1]
        popt, _ = curve_fit(
            exp_fit,
            x,
            y,
            p0=p0,
            jac=exp_jac,
            bounds=TAYLOR_BOUNDS,
            maxfev=10000
        )
        pred = exp_fit(x, *popt)
        rsq = get_rsq(y, pred)
        fit_a = float(popt[0])
        fit_b = float(popt[1])
    except Exception:
        pass # Keep rsq, fit_a, fit_b, lambda_emp as NaN

    lambda_emp = -fit_b

    # --- Quality gates (Lana's suggestion) ---
    # These gates are bypassed when skip_quality_gates=True (for plotting only).
    if not skip_quality_gates:
        if len(x) < MIN_RECOVERY_FIT_POINTS:
            return None
        if not np.isfinite(rsq) or rsq < MIN_RECOVERY_R2:
            return None

    if not np.isfinite(lambda_emp) or lambda_emp <= 0: # moved below to allow recording of rsq even if lambda_emp is bad
        return None

    prevser = raw[(raw.index < transition_date) & (raw.index >= prev_date)].copy()
    returned_to_baseline = np.nan
    if np.isfinite(prevser.values).sum() >= 3 and np.isfinite(fitting_raw_min.values).sum() >= 1:
        prev_mean = float(np.nanmean(prevser.values))
        prev_std = float(np.nanstd(prevser.values))
        returned_to_baseline = bool(np.nanmean(fitting_raw_min.tail(min(3, len(fitting_raw_min))).values) >= (prev_mean - prev_std))

    duration = int(width_months) if width_months is not None and np.isfinite(width_months) else np.nan
    if np.isfinite(duration):
        event_end_idx = min(len(raw.index) - 1, transition_idx + max(int(duration) - 1, 0))
        event_end_time = pd.Timestamp(raw.index[event_end_idx])
    else:
        event_end_time = transition_date

    fit_dates = fitting_min.index[finite]

    # Add recovery_r2 and recovery_quality_flag
    recovery_r2 = float(rsq) if np.isfinite(rsq) else np.nan
    recovery_quality_flag = bool(recovery_r2 >= MIN_RECOVERY_R2)

    return {
        "event_start_time": transition_date,
        "event_end_time": event_end_time,
        "disturbance_duration_months": duration,
        "trough_time": pd.Timestamp(fitting.index[armin]),
        "trough_value": float(fitting.iloc[armin]),
        "recovery_end_time": pd.Timestamp(fitting_min.index[-1]),
        "recovery_duration_months": int(finite.sum()),
        "returned_to_baseline": returned_to_baseline,
        "lambda_emp": float(lambda_emp) if np.isfinite(lambda_emp) and lambda_emp > 0 else np.nan, # lambda_emp must be > 0
        "lambda_emp_r2": recovery_r2,
        "recovery_quality_flag": recovery_quality_flag,
        "disturbance_magnitude": float(abs(raw_drop)) if np.isfinite(raw_drop) else np.nan,
        "transition_width_months": float(width_months) if width_months is not None and np.isfinite(width_months) else np.nan,
        "fit_a": fit_a,
        "fit_b": fit_b,
        "fit_start_time": pd.Timestamp(fit_dates[0]) if len(fit_dates) else pd.NaT,
        "fit_end_time": pd.Timestamp(fit_dates[-1]) if len(fit_dates) else pd.NaT,
    }


def build_pixel_record(da: xr.DataArray, row: pd.Series, dataset_name: str) -> dict | None:
    """
    Build one pixel record using Taylor rolling-mean + harmonic residuals.
    No da.dropna('time') is used; NaNs remain as gaps.
    """
    lat_dim, lon_dim = find_lat_lon_dims(da)
    ts = da.isel({lat_dim: int(row["lat_idx"]), lon_dim: int(row["lon_idx"])})
    if int(np.isfinite(ts.values).sum()) < MIN_POINTS:
        return None

    residual, trend, seasonal = taylor_deseason_detrend(ts)
    if residual is None:
        return None

    resid_values = residual.values.astype(float)
    if np.isfinite(resid_values).sum() < MIN_POINTS or np.nanvar(resid_values) < MIN_VARIANCE:
        return None

    resid_times = pd.to_datetime([pd.Timestamp(str(t)) for t in residual.time.values])
    raw_ser = pd.Series(ts.values.astype(float), index=pd.DatetimeIndex(resid_times))
    resid_ser = pd.Series(resid_values, index=pd.DatetimeIndex(resid_times))

    # Use the global CURRENT_K_STD_MULTIPLIER for thresholding
    transition_times_raw, widths_raw = transition_taylor(resid_values, TRANSITION_WIN_SIZE, k_std_multiplier=CURRENT_K_STD_MULTIPLIER)
    candidates = []
    for t, w in zip(list(transition_times_raw), list(widths_raw)):
        if np.isfinite(w) and w >= MIN_EVENT_LENGTH:
            candidates.append((int(t), float(w)))
    kept_times = enforce_min_separation_times([t for t, _ in candidates], MIN_SEPARATION)
    kept_lookup = {t: w for t, w in candidates if t in kept_times}

    event_rows = []          # R²-passing events only — used for benchmarking
    event_rows_all = []      # ALL attempted fits including rejected — used for plotting

    for t in kept_times:
        width = kept_lookup.get(int(t), np.nan)
        if np.isfinite(width) and width < MIN_EVENT_LENGTH:
            continue

        # First, run the fit WITHOUT quality gates to capture the attempted fit for plotting.
        event_unfiltered = fit_transition_taylor(
            transition_idx=int(t),
            detrended_resid=resid_ser,
            raw=raw_ser,
            width_months=width,
            skip_quality_gates=True,
        )
        if event_unfiltered is not None:
            event_rows_all.append(event_unfiltered)

        # Then run with quality gates for the benchmarking-quality set.
        event = fit_transition_taylor(
            transition_idx=int(t),
            detrended_resid=resid_ser,
            raw=raw_ser,
            width_months=width,
            skip_quality_gates=False,
        )
        if event is not None:
            event_rows.append(event)

    events_df = pd.DataFrame(event_rows)            # clean, used for benchmarking
    events_df_all = pd.DataFrame(event_rows_all)    # all attempts, used for plotting only

    return {
        "dataset": dataset_name,
        "pixel_id": int(row["sampled_pixel_id"]),
        "lat_idx": int(row["lat_idx"]),
        "lon_idx": int(row["lon_idx"]),
        "lat": float(row["lat"]),
        "lon": float(row["lon"]),
        "raw": ts,
        "trend": trend,
        "seasonal": seasonal,
        "anomaly": residual,
        "events_df": events_df,           # R²-filtered — used for benchmarking
        "events_df_all": events_df_all,   # all attempted fits — used for plotting only
        "n_events": int(len(events_df))
    }


def build_pixel_records_for_dataset(da: xr.DataArray, sampled_df: pd.DataFrame, dataset_name: str) -> list[dict]:
    """Build pixel records for every valid candidate pixel in the provided table."""
    records = []
    total = len(sampled_df)
    for idx_row, (_, row) in enumerate(sampled_df.iterrows(), start=1):
        if idx_row == 1 or idx_row % 100 == 0 or idx_row == total:
            print(f"  {dataset_name}: processing valid pixel {idx_row}/{total}")
        rec = build_pixel_record(da, row, dataset_name)
        if rec is not None:
            records.append(rec)
    print(f"{dataset_name}: valid pixel records={len(records)}")
    return records

## 5b. Build pixel records
Reduced test run: still potentially slow, but much lighter than the full notebook.


In [ ]:
print("Building MODIS pixel records...")
modis_records = build_pixel_records_for_dataset(modis_ndvi, modis_sampled_pixels, "MODIS")
print("Building VODCA pixel records...")
vodca_records = build_pixel_records_for_dataset(ds_vodca[vodca_var], vodca_sampled_pixels, "VODCA")
print("Building GPP pixel records...")
gpp_records = build_pixel_records_for_dataset(ds_gpp[gpp_var], gpp_sampled_pixels, "GPP")


## 5. Lana diagnostics for Taylor preprocessing

In [ ]:
print("\nSTEP 4b - Lana diagnostics: before/after detrending, valid-pixel maps, NaN maps")

import matplotlib.patches as mpatches
from matplotlib.colors import Normalize
from matplotlib.cm import ScalarMappable

DATASET_RECORDS = {
    "MODIS_NDVI": {"records": modis_records, "raw_da": modis_ndvi, "all_valid": modis_sampled_pixels, "candidates": modis_candidates, "color": "tab:blue"},
    "VODCA": {"records": vodca_records, "raw_da": ds_vodca[vodca_var], "all_valid": vodca_sampled_pixels, "candidates": vodca_candidates, "color": "tab:orange"},
    "GPP": {"records": gpp_records, "raw_da": ds_gpp[gpp_var], "all_valid": gpp_sampled_pixels, "candidates": gpp_candidates, "color": "tab:green"},
}

def _times(da):
    return pd.to_datetime([pd.Timestamp(str(t)) for t in da.time.values])

def common_ylim_from_records(records, field, qlo=1, qhi=99, pad=0.10):
    vals = []
    for rec in records:
        arr = rec[field].values.astype(float)
        vals.append(arr[np.isfinite(arr)])
    vals = np.concatenate([v for v in vals if len(v)]) if any(len(v) for v in vals) else np.array([])
    if len(vals) == 0:
        return (-1, 1)
    lo, hi = np.nanpercentile(vals, [qlo, qhi])
    if lo == hi:
        lo -= 1; hi += 1
    extra = pad * (hi - lo)
    return lo - extra, hi + extra

# ── PLOT A: Before/after detrending time series with exponential fit curves ──
def plot_record_before_after_with_fit(rec, dataset_name, color, out_path, raw_ylim=None, resid_ylim=None):
    raw   = rec["raw"]
    resid = rec["anomaly"]
    t_raw = _times(raw)
    t_res = _times(resid)

    fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=False)
    axes[0].plot(t_raw, raw.values, color=color, linewidth=1.6)
    axes[0].set_title(f"{dataset_name} original time series | lat={rec['lat']:.2f}, lon={rec['lon']:.2f}")
    axes[0].set_ylabel("Original value")
    axes[0].grid(True, alpha=0.3)
    if raw_ylim is not None:
        axes[0].set_ylim(raw_ylim)

    axes[1].plot(t_res, resid.values, color=color, linewidth=1.4, label="Residual")
    axes[1].axhline(0, color="black", linewidth=0.8, alpha=0.6)
    axes[1].set_title(f"{dataset_name} Taylor residual with exponential recovery fit")
    axes[1].set_xlabel("Time")
    axes[1].set_ylabel("Detrended value")
    axes[1].grid(True, alpha=0.3)
    if resid_ylim is not None:
        axes[1].set_ylim(resid_ylim)

    # Draw ALL attempted fits (grey dotted = rejected, red dashed = accepted)
    ev_all = rec.get("events_df_all")
    if ev_all is not None and len(ev_all) > 0:
        for _, ev_row in ev_all.iterrows():
            fit_start = pd.Timestamp(ev_row["fit_start_time"])
            fit_end   = pd.Timestamp(ev_row["fit_end_time"])
            if pd.isnull(fit_start) or pd.isnull(fit_end):
                continue
            fit_dates = pd.date_range(fit_start, fit_end, freq="MS")
            if len(fit_dates) < 2:
                continue
            fa, fb = ev_row["fit_a"], ev_row["fit_b"]
            if not (np.isfinite(fa) and np.isfinite(fb)):
                continue
            x    = np.arange(len(fit_dates), dtype=float)
            yfit = exp_fit(x, float(fa), float(fb))
            r2   = ev_row["lambda_emp_r2"]
            good = np.isfinite(r2) and r2 >= MIN_RECOVERY_R2
            lc   = "red"     if good else "#888888"
            ls   = "--"      if good else ":"
            lw   = 2.2       if good else 1.2
            lbl  = f"Accepted fit (R²={r2:.3f})" if good else f"Rejected fit (R²={r2:.3f})"
            axes[1].plot(fit_dates, yfit, linestyle=ls, color=lc, linewidth=lw, label=lbl)
            axes[1].axvline(pd.Timestamp(ev_row["trough_time"]),
                            color=lc, linewidth=0.6, alpha=0.45, linestyle=":")

    # Corner annotation
    ev_good = rec.get("events_df")
    if ev_good is not None and len(ev_good) > 0:
        best = ev_good.sort_values("lambda_emp_r2", ascending=False).iloc[0]
        txt  = (f"Best accepted fit:\n"
                f"  exponent = {best['fit_b']:.3f}\n"
                f"  R² = {best['lambda_emp_r2']:.3f}\n"
                f"  λ_emp = {best['lambda_emp']:.4f}")
        axes[1].text(0.72, 0.05, txt, transform=axes[1].transAxes, fontsize=8,
                     bbox=dict(facecolor="white", alpha=0.85, edgecolor="none"))
    elif ev_all is not None and len(ev_all) > 0:
        best_a = ev_all.sort_values("lambda_emp_r2", ascending=False).iloc[0]
        txt    = (f"No accepted fits (all rejected):\n"
                  f"  best R² = {best_a['lambda_emp_r2']:.3f}\n"
                  f"  threshold: R²≥{MIN_RECOVERY_R2}, pts≥{MIN_RECOVERY_FIT_POINTS}")
        axes[1].text(0.72, 0.05, txt, transform=axes[1].transAxes, fontsize=8,
                     color="#880000",
                     bbox=dict(facecolor="#fff5f5", alpha=0.85, edgecolor="#cc9999"))

    axes[1].legend(loc="upper left", fontsize=7, ncol=2)
    plt.tight_layout()
    plt.savefig(out_path, dpi=180, bbox_inches="tight")
    plt.show()
    plt.close()

rng = np.random.default_rng(SEED)
for dataset_name, info in DATASET_RECORDS.items():
    records = info["records"]
    if not records:
        print(dataset_name, "has no records; skipping diagnostic time-series plots")
        continue
    chosen_idx = rng.choice(len(records), size=min(N_DIAGNOSTIC_LOCATIONS, len(records)), replace=False)
    chosen_records = [records[int(i)] for i in chosen_idx]
    raw_ylim   = common_ylim_from_records(records, "raw")
    resid_ylim = common_ylim_from_records(records, "anomaly")
    for k, rec in enumerate(chosen_records, start=1):
        out = DIAG_DIR / f"{dataset_name}_before_after_taylor_location_{k}.png"
        plot_record_before_after_with_fit(rec, dataset_name, info["color"], out,
                                          raw_ylim=raw_ylim, resid_ylim=resid_ylim)
        print("Saved:", out)

# ── PLOT B: Lana Diagnostic Map 1 — exclusion / inclusion map (per dataset) ──
print("\nGenerating Lana Diagnostic Map 1: exclusion/inclusion maps...")

def plot_exclusion_map(dataset_name, raw_da, candidate_df):
    """
    Shows for one dataset:
      - Ocean (blue) vs land (grey) background
      - Study region bounding box (black dashed)
      - Non-boreal land within study region (tan)
      - Pixels inside boreal mask but excluded by data quality (orange)
      - Pixels that pass all filters = valid candidates (green)
    """
    lat_dim, lon_dim = find_lat_lon_dims(raw_da)
    lat_vals = np.asarray(raw_da[lat_dim].values, dtype=float)
    lon_vals = np.asarray(raw_da[lon_dim].values, dtype=float)
    n_lat, n_lon = len(lat_vals), len(lon_vals)

    # Recompute quality masks on the full grid
    da_t = raw_da.transpose("time", lat_dim, lon_dim)
    valid_count_2d  = np.array(da_t.count("time").values,              dtype=float)
    valid_frac_2d   = valid_count_2d / raw_da.sizes["time"]
    var_2d          = np.array(da_t.var("time", skipna=True).values,   dtype=float)

    quality_mask = (
        (valid_count_2d >= MIN_POINTS) &
        (valid_frac_2d  >= MIN_VALID_FRACTION) &
        (var_2d         >= MIN_VARIANCE)
    )

    boreal_arr = None
    if boreal_mask_on_grid is not None:
        bm = boreal_mask_on_grid
        if lat_dim in bm.dims and lon_dim in bm.dims:
            boreal_arr = np.asarray(bm.values, dtype=bool)
        elif "lat" in bm.dims and "lon" in bm.dims:
            boreal_arr = np.asarray(
                bm.interp(lat=xr.DataArray(lat_vals, dims=lat_dim),
                          lon=xr.DataArray(lon_vals, dims=lon_dim),
                          method="nearest").values, dtype=bool)

    if boreal_arr is not None:
        non_boreal          = ~boreal_arr
        boreal_excluded     = boreal_arr & ~quality_mask
        valid_candidates    = boreal_arr &  quality_mask
    else:
        non_boreal          = np.zeros((n_lat, n_lon), dtype=bool)
        boreal_excluded     = np.zeros((n_lat, n_lon), dtype=bool)
        valid_candidates    = quality_mask.astype(bool)

    fig = plt.figure(figsize=(15, 6))
    if CARTOPY_AVAILABLE:
        import cartopy.crs as ccrs
        import cartopy.feature as cfeature
        proj = ccrs.PlateCarree()
        ax   = fig.add_subplot(1, 1, 1, projection=proj)
        ax.set_extent([LON_MIN, LON_MAX, LAT_MIN, LAT_MAX], crs=proj)
        ax.add_feature(cfeature.OCEAN,     facecolor="#cce5f5", zorder=0)
        ax.add_feature(cfeature.LAND,      facecolor="#eeeeea", edgecolor="none", zorder=1)
        ax.add_feature(cfeature.COASTLINE, linewidth=0.6, edgecolor="black", zorder=5)
        ax.add_feature(cfeature.BORDERS,   linewidth=0.4, linestyle=":", edgecolor="#666666", zorder=5)
        ax.gridlines(draw_labels=True, linewidth=0.3, color="grey", alpha=0.5)
        transform = proj
    else:
        ax = fig.add_subplot(1, 1, 1)
        ax.set_xlim(LON_MIN, LON_MAX); ax.set_ylim(LAT_MIN, LAT_MAX)
        ax.set_facecolor("#cce5f5")
        ax.grid(True, alpha=0.3)
        transform = None

    def _pmesh(mask_2d, color):
        arr = np.where(mask_2d, 1.0, np.nan)
        cmap_single = plt.matplotlib.colors.ListedColormap([color])
        kw = dict(cmap=cmap_single, shading="auto", vmin=0.5, vmax=1.5, zorder=3, alpha=0.85)
        if transform is not None:
            ax.pcolormesh(lon_vals, lat_vals, arr, transform=transform, **kw)
        else:
            ax.pcolormesh(lon_vals, lat_vals, arr, **kw)

    _pmesh(non_boreal,       "#d4c9a8")   # non-boreal land — tan
    _pmesh(boreal_excluded,  "#e07b39")   # boreal but data-quality excluded — orange
    _pmesh(valid_candidates, "#2d7a3a")   # valid candidate pixels — green

    # Re-draw ocean above the raster mask so ocean is blue, not tan.
    # This fixes the priority problem where non-boreal=False over ocean could otherwise cover ocean.
    if CARTOPY_AVAILABLE:
        ax.add_feature(cfeature.OCEAN, facecolor="#cce5f5", zorder=4)
        ax.add_feature(cfeature.COASTLINE, linewidth=0.6, edgecolor="black", zorder=5)

    # Study region bounding box
    rect = plt.matplotlib.patches.Rectangle(
        (LON_MIN, LAT_MIN), LON_MAX - LON_MIN, LAT_MAX - LAT_MIN,
        linewidth=1.8, edgecolor="black", facecolor="none",
        linestyle="--", transform=(transform if transform else ax.transData), zorder=6)
    ax.add_patch(rect)

    legend_handles = [
        mpatches.Patch(facecolor="#cce5f5", edgecolor="grey",   label="Ocean"),
        mpatches.Patch(facecolor="#eeeeea", edgecolor="grey",   label="Land (outside study region)"),
        mpatches.Patch(facecolor="#d4c9a8", edgecolor="none",   label="Not boreal forest"),
        mpatches.Patch(facecolor="#e07b39", edgecolor="none",   label="Boreal — excluded (data quality)"),
        mpatches.Patch(facecolor="#2d7a3a", edgecolor="none",   label="Boreal — valid candidate"),
        mpatches.Patch(facecolor="none",    edgecolor="black",
                       linestyle="--", linewidth=1.5,           label="Study region boundary"),
    ]
    ax.legend(handles=legend_handles, loc="lower left", fontsize=7.5, framealpha=0.92)
    ax.set_title(f"{dataset_name}: Study region, boreal mask, and data-quality exclusions", fontsize=11)

    n_excl  = int(boreal_excluded.sum())
    n_valid = int(valid_candidates.sum())
    ax.text(0.01, 0.97,
            f"Valid candidates: {n_valid} | Excluded (quality): {n_excl}",
            transform=ax.transAxes, fontsize=8, va="top",
            bbox=dict(facecolor="white", alpha=0.7, edgecolor="none"))

    fname = DIAG_DIR / f"lana_map1_exclusions_{dataset_name}.png"
    plt.tight_layout()
    plt.savefig(fname, dpi=150, bbox_inches="tight")
    plt.show()
    plt.close()
    print(f"Saved: {fname}  [valid={n_valid}, excluded={n_excl}]")

for ds_name, info in DATASET_RECORDS.items():
    plot_exclusion_map(ds_name, info["raw_da"], info["candidates"])

# ── PLOT C: Lana Diagnostic Map 2 — all valid pixels by NaN fraction ──
print("\nGenerating Lana Diagnostic Map 2: all valid candidate pixels coloured by NaN fraction...")

def plot_all_valid_pixel_map(dataset_name, raw_da, candidate_df):
    """
    Shows for one dataset:
      - Ocean/land background + study region outline + boreal mask (faint green)
      - Every valid candidate pixel used in the main run
      - Candidate pixels coloured by NaN fraction (viridis 0–1)

    This replaces the old stratified-sampling map because the full run no longer samples pixels.
    """
    lat_dim, lon_dim = find_lat_lon_dims(raw_da)
    lat_vals = np.asarray(raw_da[lat_dim].values, dtype=float)
    lon_vals = np.asarray(raw_da[lon_dim].values, dtype=float)

    fig = plt.figure(figsize=(15, 6))
    if CARTOPY_AVAILABLE:
        import cartopy.crs as ccrs
        import cartopy.feature as cfeature
        proj      = ccrs.PlateCarree()
        ax        = fig.add_subplot(1, 1, 1, projection=proj)
        transform = proj
        ax.set_extent([LON_MIN, LON_MAX, LAT_MIN, LAT_MAX], crs=proj)
        ax.add_feature(cfeature.OCEAN,     facecolor="#cce5f5", zorder=0)
        ax.add_feature(cfeature.LAND,      facecolor="#eeeeea", edgecolor="none", zorder=1)
        ax.add_feature(cfeature.COASTLINE, linewidth=0.6, edgecolor="black", zorder=5)
        ax.add_feature(cfeature.BORDERS,   linewidth=0.4, linestyle=":", edgecolor="#666666", zorder=5)
        ax.gridlines(draw_labels=True, linewidth=0.3, color="grey", alpha=0.5)
    else:
        ax        = fig.add_subplot(1, 1, 1)
        transform = None
        ax.set_xlim(LON_MIN, LON_MAX); ax.set_ylim(LAT_MIN, LAT_MAX)
        ax.set_facecolor("#cce5f5")
        ax.grid(True, alpha=0.3)

    # Boreal mask faint green fill
    if boreal_mask_on_grid is not None:
        try:
            boreal_arr = np.asarray(boreal_mask_on_grid.values, dtype=float)
            boreal_plot = np.where(boreal_arr == 1, 1.0, np.nan)
            kw = dict(cmap=plt.matplotlib.colors.ListedColormap(["#b2dfb5"]),
                      shading="auto", vmin=0.5, vmax=1.5, zorder=2, alpha=0.35)
            if transform is not None:
                ax.pcolormesh(lon_vals, lat_vals, boreal_plot, transform=transform, **kw)
                # redraw ocean so it stays blue
                ax.add_feature(cfeature.OCEAN, facecolor="#cce5f5", zorder=3)
                ax.add_feature(cfeature.COASTLINE, linewidth=0.6, edgecolor="black", zorder=5)
            else:
                ax.pcolormesh(lon_vals, lat_vals, boreal_plot, **kw)
        except Exception as e:
            print(f"Could not draw boreal mask for {dataset_name}: {e}")

    # Study region bounding box
    rect = plt.matplotlib.patches.Rectangle(
        (LON_MIN, LAT_MIN), LON_MAX - LON_MIN, LAT_MAX - LAT_MIN,
        linewidth=1.8, edgecolor="black", facecolor="none",
        linestyle="--", transform=(transform if transform else ax.transData), zorder=6)
    ax.add_patch(rect)

    # All valid pixels coloured by NaN fraction
    if len(candidate_df) > 0 and "nan_fraction" in candidate_df.columns:
        nf   = candidate_df["nan_fraction"].values
        norm = Normalize(vmin=0.0, vmax=1.0)
        cmap = plt.cm.viridis
        sc_kw = dict(c=nf, cmap=cmap, norm=norm, s=14,
                     edgecolors="black", linewidths=0.15, zorder=7)
        if transform is not None:
            ax.scatter(candidate_df["lon"].values, candidate_df["lat"].values,
                       transform=transform, **sc_kw)
        else:
            ax.scatter(candidate_df["lon"].values, candidate_df["lat"].values, **sc_kw)
        sm = ScalarMappable(cmap=cmap, norm=norm); sm.set_array([])
        cbar = plt.colorbar(sm, ax=ax, shrink=0.75, pad=0.02)
        cbar.set_label("NaN fraction at valid candidate pixel", fontsize=9)

    legend_handles = [
        mpatches.Patch(facecolor="#cce5f5", edgecolor="grey",   label="Ocean"),
        mpatches.Patch(facecolor="#eeeeea", edgecolor="grey",   label="Land"),
        mpatches.Patch(facecolor="#b2dfb5", edgecolor="none",   label="Boreal forest mask"),
        mpatches.Patch(facecolor="none",    edgecolor="black",
                       linestyle="--", linewidth=1.5,           label="Study region boundary"),
    ]
    ax.legend(handles=legend_handles, loc="lower left", fontsize=7.5, framealpha=0.92)
    ax.set_title(
        f"{dataset_name}: all valid candidate pixels used in full run (n={len(candidate_df)}), coloured by NaN fraction",
        fontsize=11)

    fname = DIAG_DIR / f"lana_map2_all_valid_pixels_{dataset_name}.png"
    plt.tight_layout()
    plt.savefig(fname, dpi=150, bbox_inches="tight")
    plt.show()
    plt.close()
    print(f"Saved: {fname}  [n_valid_pixels={len(candidate_df)}]")

for ds_name, info in DATASET_RECORDS.items():
    plot_all_valid_pixel_map(ds_name, info["raw_da"], info["all_valid"])

# ── PLOT D: Original-style NaN fraction and sampled pixel maps (kept for continuity) ──
def setup_map_ax(title):
    if CARTOPY_AVAILABLE:
        ax = plt.axes(projection=ccrs.PlateCarree())
        ax.set_extent([LON_MIN, LON_MAX, LAT_MIN, LAT_MAX], crs=ccrs.PlateCarree())
        ax.add_feature(cfeature.OCEAN, facecolor="#d9eef7", zorder=0)
        ax.add_feature(cfeature.LAND,  facecolor="#f0f0e8", edgecolor="black", linewidth=0.4, zorder=0)
        ax.coastlines(resolution="50m", linewidth=0.6)
        gl = ax.gridlines(draw_labels=True, linewidth=0.2, color="gray", alpha=0.4)
        gl.top_labels = False; gl.right_labels = False
        ax.set_title(title)
        return ax
    else:
        ax = plt.gca()
        ax.set_xlim(LON_MIN, LON_MAX); ax.set_ylim(LAT_MIN, LAT_MAX)
        ax.set_xlabel("Longitude"); ax.set_ylabel("Latitude")
        ax.set_title(title); ax.grid(True, alpha=0.3)
        return ax

def plot_nan_fraction_map_cartopy(records, dataset_name):
    da_template = DATASET_RECORDS[dataset_name]["raw_da"]
    nan_grid = np.full((da_template.sizes["lat"], da_template.sizes["lon"]), np.nan, dtype=float)
    for rec in records:
        nan_grid[int(rec["lat_idx"]), int(rec["lon_idx"])] = np.isnan(rec["anomaly"].values).mean()
    lon2d, lat2d = np.meshgrid(da_template.lon.values, da_template.lat.values)
    plt.figure(figsize=(12, 6))
    ax   = setup_map_ax(f"{dataset_name}: NaN fraction at valid pixels after Taylor detrending")
    cmap = plt.cm.viridis.copy(); cmap.set_bad(color=(1,1,1,0))
    im   = ax.pcolormesh(lon2d, lat2d, nan_grid, cmap=cmap, vmin=0, vmax=1,
                          transform=ccrs.PlateCarree() if CARTOPY_AVAILABLE else None,
                          shading="auto") if CARTOPY_AVAILABLE else            ax.pcolormesh(lon2d, lat2d, nan_grid, cmap=cmap, vmin=0, vmax=1, shading="auto")
    plt.colorbar(im, ax=ax, label="Fraction NaN")
    out = DIAG_DIR / f"{dataset_name}_taylor_nan_fraction_map_with_land.png"
    plt.tight_layout(); plt.savefig(out, dpi=220); plt.show(); print("Saved:", out)

def plot_sampled_pixel_map_cartopy(records, dataset_name):
    plt.figure(figsize=(12, 6))
    ax   = setup_map_ax(f"{dataset_name}: all valid pixels")
    lons = [r["lon"] for r in records]
    lats = [r["lat"] for r in records]
    if CARTOPY_AVAILABLE:
        ax.scatter(lons, lats, s=30, c="red", edgecolor="black", linewidth=0.4,
                   transform=ccrs.PlateCarree(), label="Valid pixels", zorder=5)
    else:
        ax.scatter(lons, lats, s=30, c="red", edgecolor="black", linewidth=0.4, label="Valid pixels")
    ax.legend(loc="upper right")
    out = DIAG_DIR / f"{dataset_name}_taylor_sampled_pixel_map_with_land.png"
    plt.tight_layout(); plt.savefig(out, dpi=220); plt.show(); print("Saved:", out)

for dataset_name, info in DATASET_RECORDS.items():
    plot_nan_fraction_map_cartopy(info["records"], dataset_name)
    plot_sampled_pixel_map_cartopy(info["records"], dataset_name)

# Summary CSV
summary_rows = []
for dataset_name, info in DATASET_RECORDS.items():
    for rec in info["records"]:
        raw_vals = rec["raw"].values.astype(float)
        res_vals = rec["anomaly"].values.astype(float)
        summary_rows.append({
            "dataset":   dataset_name,
            "pixel_id":  rec["pixel_id"],
            "lat":       rec["lat"],
            "lon":       rec["lon"],
            "raw_valid_fraction":      float(np.isfinite(raw_vals).mean()),
            "residual_valid_fraction": float(np.isfinite(res_vals).mean()),
            "valid_fraction_loss":     float(np.isfinite(raw_vals).mean() - np.isfinite(res_vals).mean()),
            "n_events":  rec["n_events"],
        })
summary_df   = pd.DataFrame(summary_rows)
summary_path = DIAG_DIR / "taylor_nan_and_sampled_pixel_summary.csv"
summary_df.to_csv(summary_path, index=False)
print("Saved diagnostic summary:", summary_path)
if len(summary_df) > 0:
    display(summary_df.groupby("dataset")[["raw_valid_fraction","residual_valid_fraction","valid_fraction_loss","n_events"]].describe())


## 6. Event-level theoretical metrics
Compute local variance, AC1, and AR-based lambda around each event.

In [ ]:
print("\nSTEP 5 - Event-level theoretical metrics (including a_GLSAR and Taylor lambda_var)")

def get_event_window(residual_da: xr.DataArray, center_time: pd.Timestamp, window_months: int = 60, mode: str = "pre"):
    if mode == "center":
        start = center_time - pd.DateOffset(months=window_months // 2)
        end = center_time + pd.DateOffset(months=window_months // 2)
    else:
        end = center_time
        start = center_time - pd.DateOffset(months=window_months)
    # Do not drop time values. Keep gaps; metric functions handle finite pairs.
    seg = residual_da.sel(time=slice(start, end))
    return seg

def lambda_variance_taylor(values: np.ndarray, dt: float = 1.0) -> float:
    """Taylor/Smith variance-based recovery rate estimate."""
    x = np.asarray(values, dtype=float)
    if np.isfinite(x).sum() < 5 or np.nanvar(x) <= MIN_VARIANCE:
        return np.nan
    dx = (x[1:] - x[:-1]) / dt
    x0 = x[:-1]
    mask = np.isfinite(x0) & np.isfinite(dx)
    if mask.sum() < 5:
        return np.nan
    lamb = linregress(x0[mask], dx[mask]).slope
    diff = dx[mask] - lamb * x0[mask]
    sigma = np.nanstd(diff) * np.sqrt(dt)
    var = np.nanvar(x)
    arg = 1 - sigma**2 / var
    if not np.isfinite(arg) or arg <= 0:
        return np.nan
    return float(0.5 * np.log(arg) / dt)


def compute_local_theoretical_metrics(anomaly_da: xr.DataArray, center_time: pd.Timestamp, window_months: int = 60, mode: str = "pre"):
    """
    Compute theoretical metrics on Taylor residuals.
    Default is a pre-event window. NaNs remain in the window and are handled by the metric functions.
    """
    seg = get_event_window(anomaly_da, center_time=center_time, window_months=window_months, mode=mode)
    values = seg.values.astype(float)
    npts = int(np.isfinite(values).sum())
    if npts < max(24, window_months // 2):
        return {
            "variance": np.nan,
            "ac1": np.nan,
            "lambda_ar1": np.nan,
            "lambda_var": np.nan,
            "a_glsar": np.nan,
            "window_points": npts
        }

    if np.nanvar(values) < MIN_VARIANCE:
        return {
            "variance": np.nan,
            "ac1": np.nan,
            "lambda_ar1": np.nan,
            "lambda_var": np.nan,
            "a_glsar": np.nan,
            "window_points": npts
        }

    return {
        "variance": float(np.nanvar(values)),
        "ac1": lag1_autocorrelation(values),
        "lambda_ar1": ar1_lambda(values),
        "lambda_var": lambda_variance_taylor(values),
        "a_glsar": glsar_a(values),
        "window_points": npts
    }


def build_event_level_table(records: list[dict], dataset_name: str) -> pd.DataFrame:
    expected_cols = [
        "dataset", "pixel_id", "lat", "lon", "event_id",
        "event_start_time", "event_end_time", "trough_time", "recovery_end_time",
        "disturbance_duration_months", "recovery_duration_months", "returned_to_baseline",
        "disturbance_magnitude", "transition_width_months", "lambda_emp", "lambda_emp_r2",
        "fit_a", "fit_b", "fit_start_time", "fit_end_time",
        "variance", "ac1", "lambda_ar1", "lambda_var", "a_glsar", "window_points",
    ]

    rows = []
    total = len(records)
    for idx_rec, rec in enumerate(records, start=1):
        if idx_rec == 1 or idx_rec % 25 == 0 or idx_rec == total:
            print(f"  {dataset_name}: event metrics for pixel record {idx_rec}/{total}")

        anomaly_da = rec["anomaly"]
        events_df = rec["events_df"]
        if events_df is None or len(events_df) == 0:
            continue

        for event_idx, event in events_df.iterrows():
            local = compute_local_theoretical_metrics(
                anomaly_da=anomaly_da,
                center_time=pd.Timestamp(event["trough_time"]),
                window_months=WINDOW_SIZE,
                mode=EVENT_WINDOW_MODE,
            )
            rows.append({
                "dataset": rec["dataset"],
                "pixel_id": rec["pixel_id"],
                "lat": rec["lat"],
                "lon": rec["lon"],
                "event_id": int(event_idx),
                "event_start_time": event["event_start_time"],
                "event_end_time": event["event_end_time"],
                "trough_time": event["trough_time"],
                "recovery_end_time": event["recovery_end_time"],
                "disturbance_duration_months": event["disturbance_duration_months"],
                "recovery_duration_months": event["recovery_duration_months"],
                "returned_to_baseline": event["returned_to_baseline"],
                "disturbance_magnitude": event["disturbance_magnitude"],
                "transition_width_months": event.get("transition_width_months", np.nan),
                "lambda_emp": event["lambda_emp"],
                "lambda_emp_r2": event["lambda_emp_r2"],
                "fit_a": event.get("fit_a", np.nan),
                "fit_b": event.get("fit_b", np.nan),
                "fit_start_time": event.get("fit_start_time", pd.NaT),
                "fit_end_time": event.get("fit_end_time", pd.NaT),
                "variance": local["variance"],
                "ac1": local["ac1"],
                "lambda_ar1": local["lambda_ar1"],
                "lambda_var": local["lambda_var"],
                "a_glsar": local["a_glsar"],
                "window_points": local["window_points"],
            })

    return pd.DataFrame(rows, columns=expected_cols)

## 6b. Build event-level benchmark table

In [ ]:
print("Building event-level table for MODIS...")
events_modis = build_event_level_table(modis_records, "MODIS")
print("Building event-level table for VODCA...")
events_vodca = build_event_level_table(vodca_records, "VODCA")
print("Building event-level table for GPP...")
events_gpp = build_event_level_table(gpp_records, "GPP")

events_all = pd.concat([events_modis, events_vodca, events_gpp], ignore_index=True)
events_all.to_csv(OUTPUT_DIR / "event_level_benchmark_table.csv", index=False)

print("MODIS events:", len(events_modis))
print("VODCA events:", len(events_vodca))
print("GPP events:", len(events_gpp))
print("Total valid events:", len(events_all))
print("Columns:", list(events_all.columns))
print(events_all.head())


## 7. Pixel-level summary table

In [ ]:
print("\nSTEP 6 - Pixel-level summary table")

def build_pixel_summary(records: list[dict]) -> pd.DataFrame:
    rows = []
    for rec in records:
        values = rec["anomaly"].values.astype(float)
        rows.append({
            "dataset": rec["dataset"],
            "pixel_id": rec["pixel_id"],
            "lat": rec["lat"],
            "lon": rec["lon"],
            "n_events": rec["n_events"],
            "variance_full_series": float(np.var(values)) if len(values) > 0 else np.nan,
            "ac1_full_series": lag1_autocorrelation(values),
            "lambda_ar1_full_series": ar1_lambda(values)
        })
    return pd.DataFrame(rows)

print("Building pixel summary table...")
pixel_summary = pd.concat([
    build_pixel_summary(modis_records),
    build_pixel_summary(vodca_records),
    build_pixel_summary(gpp_records)
], ignore_index=True)

pixel_summary.to_csv(OUTPUT_DIR / "pixel_summary.csv", index=False)
print("Saved pixel summary table")


## 8. Benchmarking theoretical metrics against empirical recovery

In [ ]:
print("\nSTEP 7 - Proper benchmarking")

def benchmark_metric(df: pd.DataFrame, metric_col: str, metric_name: str) -> dict:
    """
    Benchmark one theoretical metric against lambda_emp.
    """
    tmp = df[["lambda_emp", metric_col]].dropna()
    n_events = len(tmp)

    if n_events < 3:
        return {
            "metric": metric_name,
            "n_events": n_events,
            "spearman_r": np.nan,
            "spearman_p": np.nan,
            "pearson_r": np.nan,
            "pearson_p": np.nan,
            "mae_after_sign_alignment": np.nan,
            "coverage": n_events / max(len(df), 1)
        }

    sp_r, sp_p, _ = safe_spearman(tmp[metric_col], tmp["lambda_emp"])
    pr_r, pr_p, _ = safe_pearson(tmp[metric_col], tmp["lambda_emp"])

    aligned_metric = tmp[metric_col].copy()
    # Align signs for MAE calculation: variance and ac1 are expected to be negatively correlated with lambda_emp
    if metric_name in ["variance", "ac1"]:
        aligned_metric = -aligned_metric

    mae = float(np.mean(np.abs(aligned_metric - tmp["lambda_emp"]))) # MAE is always positive

    return {
        "metric": metric_name,
        "n_events": n_events,
        "spearman_r": sp_r,
        "spearman_p": sp_p,
        "pearson_r": pr_r,
        "pearson_p": pr_p,
        "mae_after_sign_alignment": mae,
        "coverage": n_events / max(len(df), 1)
    }


def benchmark_dataset(df: pd.DataFrame, dataset_name: str) -> pd.DataFrame:
    if "dataset" not in df.columns:
        return pd.DataFrame(columns=[
            "metric", "n_events", "n_pixels", "coverage", "spearman_r", "spearman_p",
            "pearson_r", "pearson_p", "mae_after_sign_alignment"
        ])

    dataset_df = df[df["dataset"] == dataset_name].copy()
    n_pixels = dataset_df["pixel_id"].nunique()

    results = []
    for metric_col, metric_name in [
        ("variance", "variance"),
        ("ac1", "ac1"),
        ("lambda_ar1", "lambda_ar1"),
        ("lambda_var", "lambda_var"),
        ("a_glsar", "a_glsar"),
    ]:
        out = benchmark_metric(dataset_df, metric_col, metric_name)
        out["dataset"] = dataset_name
        out["n_pixels"] = n_pixels # Add n_pixels here
        results.append(out)

    return pd.DataFrame(results)

# --- Report how many events were retained vs. filtered by the R² and min-point thresholds ---
print("\n--- Recovery fit quality summary (after R² and min-point filtering) ---")
if len(events_all) > 0:
    for ds_name in ["MODIS", "VODCA", "GPP"]:
        ds_events = events_all[events_all["dataset"] == ds_name]
        n_total = len(ds_events)
        n_good_r2 = int((ds_events["lambda_emp_r2"] >= MIN_RECOVERY_R2).sum()) if n_total > 0 else 0
        r2_values = ds_events["lambda_emp_r2"].dropna()
        median_r2 = float(r2_values.median()) if len(r2_values) > 0 else float("nan")
        print(f"  {ds_name}: {n_total} events retained | median R²={median_r2:.3f} | events with R²>={MIN_RECOVERY_R2}: {n_good_r2}")
else:
    print("  No events available.")
print("---------------------------------------------------------------------\n")


print("Benchmarking MODIS, VODCA, and GPP...")

# Filter events for quality before benchmarking if a flag is available
# This also means that 'n_events' and 'coverage' below will refer to high-quality events.
if 'recovery_quality_flag' in events_all.columns:
    events_benchmarking = events_all[events_all['recovery_quality_flag']].copy()
    print(f"Benchmarking {len(events_benchmarking)} high-quality events (out of {len(events_all)} total events).")
else:
    events_benchmarking = events_all.copy()
    print(f"Benchmarking {len(events_benchmarking)} events (recovery_quality_flag not available).")


if len(events_benchmarking) == 0:
    print("No valid events available for benchmarking in this run.")
    benchmark_results = pd.DataFrame(columns=[
        "metric", "n_events", "n_pixels", "coverage", "spearman_r", "spearman_p",
        "pearson_r", "pearson_p", "mae_after_sign_alignment", "dataset"
    ])
else:
    benchmark_results = pd.concat([
        benchmark_dataset(events_benchmarking, "MODIS"),
        benchmark_dataset(events_benchmarking, "VODCA"),
        benchmark_dataset(events_benchmarking, "GPP"),
    ], ignore_index=True)

benchmark_results.to_csv(OUTPUT_DIR / "benchmark_results.csv", index=False)
print(benchmark_results.to_string())

## 9. Ranking dataset-metric combinations

In [ ]:
print("\nSTEP 8 - Ranking dataset-metric combinations")

def minmax_scale(series: pd.Series) -> pd.Series:
    s = series.astype(float).copy()
    finite = np.isfinite(s)
    if finite.sum() == 0:
        return pd.Series(np.nan, index=series.index)
    mn = np.nanmin(s[finite])
    mx = np.nanmax(s[finite])
    if mx == mn:
        out = pd.Series(1.0, index=series.index)
        out[~finite] = np.nan
        return out
    out = (s - mn) / (mx - mn)
    out[~finite] = np.nan
    return out


ranking = benchmark_results.copy()

# Ensure all metrics are present in the dataframe before attempting to score them
if 'spearman_r' in ranking.columns:
    ranking["score_abs_spearman"] = minmax_scale(np.abs(ranking["spearman_r"]))
else:
    ranking["score_abs_spearman"] = np.nan

if 'pearson_r' in ranking.columns:
    ranking["score_abs_pearson"] = minmax_scale(np.abs(ranking["pearson_r"]))
else:
    ranking["score_abs_pearson"] = np.nan

if 'mae_after_sign_alignment' in ranking.columns:
    # Lower MAE is better, so (1 - normalized_mae) is better
    ranking["score_inverse_normalized_mae"] = 1 - minmax_scale(ranking["mae_after_sign_alignment"])
else:
    ranking["score_inverse_normalized_mae"] = np.nan

if 'coverage' in ranking.columns:
    ranking["score_coverage"] = minmax_scale(ranking["coverage"])
else:
    ranking["score_coverage"] = np.nan

# Combine scores. Fillna(0) for components in case of NaNs, so they don't drag down the average score for other valid components.
ranking["overall_score"] = ranking[[
    "score_abs_spearman",
    "score_abs_pearson",
    "score_inverse_normalized_mae",
    "score_coverage"
]].fillna(0).mean(axis=1)

ranking = ranking.sort_values("overall_score", ascending=False).reset_index(drop=True)
ranking.to_csv(OUTPUT_DIR / "dataset_metric_ranking.csv", index=False)

print(ranking.to_string())

best_row = ranking.iloc[0].to_dict() if len(ranking) > 0 else None
print("\nBest combination:", best_row)
print("Saved ranking table")

## 10. Sliding-window analysis functions

In [ ]:
print("\nSTEP 9 - Sliding window analysis")

def compute_rolling_metrics_for_record(rec: dict) -> pd.DataFrame:
    anomaly_da = rec["anomaly"]
    values = anomaly_da.values.astype(float)
    times = pd.to_datetime([pd.Timestamp(str(t)) for t in anomaly_da.time.values])

    rows = []
    if len(values) < WINDOW_SIZE:
        return pd.DataFrame(rows)

    for i in range(0, len(values) - WINDOW_SIZE + 1, WINDOW_STEP):
        seg = values[i:i + WINDOW_SIZE]
        seg_times = times[i:i + WINDOW_SIZE]

        if np.isfinite(seg).sum() < max(24, WINDOW_SIZE // 2):
            continue
        if np.nanvar(seg) < MIN_VARIANCE:
            continue

        if EVENT_WINDOW_MODE == "pre":
            center_time = pd.Timestamp(seg_times[-1])
        else:
            center_time = pd.Timestamp(seg_times[len(seg_times) // 2])

        rows.append({
            "dataset": rec["dataset"],
            "pixel_id": rec["pixel_id"],
            "lat": rec["lat"],
            "lon": rec["lon"],
            "time": center_time,
            "variance": float(np.nanvar(seg)),
            "ac1": lag1_autocorrelation(seg),
            "lambda_ar1": ar1_lambda(seg),
            "lambda_var": lambda_variance_taylor(seg),
            "a_glsar": glsar_a(seg),
        })

    return pd.DataFrame(rows)

## 10b. Run sliding-window analysis
Full run: uses `WINDOW_SIZE = 60` on the full monthly record.


In [ ]:
def compute_rolling_metrics_for_dataset(records: list[dict], dataset_name: str) -> pd.DataFrame:
    parts = []
    total = len(records)
    for idx_rec, rec in enumerate(records, start=1):
        if idx_rec == 1 or idx_rec % 25 == 0 or idx_rec == total:
            print(f"  {dataset_name}: rolling metrics for pixel record {idx_rec}/{total}")
        parts.append(compute_rolling_metrics_for_record(rec))
    if len(parts) == 0:
        return pd.DataFrame()
    return pd.concat(parts, ignore_index=True)

print("Computing rolling metrics for MODIS...")
rolling_modis = compute_rolling_metrics_for_dataset(modis_records, "MODIS")
print("Computing rolling metrics for VODCA...")
rolling_vodca = compute_rolling_metrics_for_dataset(vodca_records, "VODCA")
print("Computing rolling metrics for GPP...")
rolling_gpp = compute_rolling_metrics_for_dataset(gpp_records, "GPP")

rolling_all = pd.concat([
    rolling_modis,
    rolling_vodca,
    rolling_gpp,
], ignore_index=True)

rolling_all.to_csv(OUTPUT_DIR / "rolling_metrics_all.csv", index=False)
print("Rolling rows:", len(rolling_all))


## 11. Figures and final outputs
Rolling metrics are summarized as medians across all valid pixels.


In [ ]:
print("\nSTEP 10 - Figures")

def save_line_plot(df: pd.DataFrame, value_col: str, title: str, ylabel: str, filename: str):
    plot_df = df.groupby(["dataset", "time"])[value_col].median().reset_index()

    plt.figure(figsize=(12, 6))
    for dataset in plot_df["dataset"].unique():
        tmp = plot_df[plot_df["dataset"] == dataset]
        plt.plot(tmp["time"], tmp[value_col], linewidth=2, label=dataset)

    plt.title(title)
    plt.xlabel("Time")
    plt.ylabel(ylabel)
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / filename, dpi=300)
    plt.close()


if len(rolling_all) > 0:
    print("Saving rolling-metric figures...")
    rolling_all["time"] = pd.to_datetime(rolling_all["time"])

    # Clip edge windows for plotting only
    # The calculation is a bit tricky due to pixel-wise dataframes concatenated.
    # For simplicity, we'll assume the time series for each pixel is long enough
    # to apply this directly, or handle potential empty slices.
    rolling_plot_df = rolling_all.copy()
    if CLIP_ROLLING_EDGE_WINDOWS > 0 and len(rolling_plot_df) > 2 * CLIP_ROLLING_EDGE_WINDOWS:
        # Group by pixel_id and dataset to apply clipping correctly to each individual series
        def _clip_df_group(group_df):
            if len(group_df) > 2 * CLIP_ROLLING_EDGE_WINDOWS:
                return group_df.iloc[CLIP_ROLLING_EDGE_WINDOWS:-CLIP_ROLLING_EDGE_WINDOWS]
            return pd.DataFrame() # Return empty if not enough data after clipping

        rolling_plot_df = rolling_plot_df.groupby(['dataset', 'pixel_id']).apply(_clip_df_group).reset_index(drop=True)

    if len(rolling_plot_df) > 0:
        save_line_plot(
            rolling_plot_df,
            value_col="ac1",
            title="Rolling AC1 (median across all valid pixels)",
            ylabel="AC1",
            filename="figure_rolling_ac1.png"
        )

        save_line_plot(
            rolling_plot_df,
            value_col="variance",
            title="Rolling variance (median across all valid pixels)",
            ylabel="Variance",
            filename="figure_rolling_variance.png"
        )

        save_line_plot(
            rolling_plot_df,
            value_col="lambda_ar1",
            title="Rolling AR-based recovery rate (median across all valid pixels)",
            ylabel="AR-based lambda",
            filename="figure_rolling_lambda_ar1.png"
        )

        save_line_plot(
            rolling_plot_df,
            value_col="a_glsar",
            title="Rolling a_GLSAR (median across all valid pixels)",
            ylabel="a_GLSAR",
            filename="figure_rolling_a_glsar.png"
        )

        save_line_plot(
            rolling_plot_df,
            value_col="lambda_var",
            title="Rolling variance-based recovery rate (median across all valid pixels)",
            ylabel="Variance-based lambda",
            filename="figure_rolling_lambda_var.png"
        )
    else:
        print("Not enough data in rolling metrics after clipping for plotting.")

if len(ranking) > 0:
    print("Saving benchmark ranking figure...")
    plt.figure(figsize=(12, 6))
    labels = ranking["dataset"] + " | " + ranking["metric"]
    plt.bar(labels, ranking["overall_score"])
    plt.xticks(rotation=45, ha="right")
    plt.ylabel("Overall benchmark score")
    plt.title("Ranking of dataset-metric combinations")
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "figure_dataset_metric_ranking.png", dpi=300)
    plt.close()

event_counts = events_all.groupby("dataset").size().reset_index(name="n_events")
event_counts.to_csv(OUTPUT_DIR / "event_counts_by_dataset.csv", index=False)

print("\nSaved all-valid-pixel outputs in:", OUTPUT_DIR)
print("Pipeline completed successfully.")

## 12. Event maps with land/ocean boundary
These maps show all valid candidate pixels in grey and detected disturbance events in red.


In [ ]:
print("\nSTEP 10b - Event maps with land/ocean boundary")

def setup_event_map_ax(title):
    if CARTOPY_AVAILABLE:
        ax = plt.axes(projection=ccrs.PlateCarree())
        ax.set_extent([LON_MIN, LON_MAX, LAT_MIN, LAT_MAX], crs=ccrs.PlateCarree())
        ax.add_feature(cfeature.OCEAN, facecolor="#d9eef7", zorder=0)
        ax.add_feature(cfeature.LAND, facecolor="#f0f0e8", edgecolor="black", linewidth=0.4, zorder=0)
        ax.coastlines(resolution="50m", linewidth=0.7)
        gl = ax.gridlines(draw_labels=True, linewidth=0.2, color="black", alpha=0.4)
        gl.top_labels = False
        gl.right_labels = False
        ax.set_title(title)
        return ax
    else:
        ax = plt.gca()
        ax.set_xlim(LON_MIN, LON_MAX); ax.set_ylim(LAT_MIN, LAT_MAX)
        ax.set_xlabel("Longitude"); ax.set_ylabel("Latitude")
        ax.set_title(title)
        ax.grid(True, alpha=0.3)
        return ax


def plot_event_detection_map(events_df, valid_df, dataset_name):
    plt.figure(figsize=(12, 6))
    ax = setup_event_map_ax(f"{dataset_name}: detected disturbance events over land boundary (all valid pixels)")

    # All valid candidate pixels in grey for context.
    if len(valid_df) > 0:
        if CARTOPY_AVAILABLE:
            ax.scatter(valid_df["lon"], valid_df["lat"], s=12, c="darkgray", edgecolor="none",
                       transform=ccrs.PlateCarree(), label="All valid pixels", zorder=3, alpha=0.65)
        else:
            ax.scatter(valid_df["lon"], valid_df["lat"], s=12, c="darkgray", edgecolor="none",
                       label="All valid pixels", alpha=0.65)

    if events_df is not None and len(events_df) > 0:
        if CARTOPY_AVAILABLE:
            ax.scatter(events_df["lon"], events_df["lat"], s=45, c="red", edgecolor="black", linewidth=0.4,
                       transform=ccrs.PlateCarree(), label="Detected events", zorder=5)
        else:
            ax.scatter(events_df["lon"], events_df["lat"], s=45, c="red", edgecolor="black", linewidth=0.4,
                       label="Detected events")
    ax.legend(loc="upper right")
    out = OUTPUT_DIR / f"figure_event_detection_map_{dataset_name}.png"
    plt.tight_layout(); plt.savefig(out, dpi=300); plt.show(); print("Saved:", out)

for dataset_name, valid_df in [("MODIS", modis_sampled_pixels), ("VODCA", vodca_sampled_pixels), ("GPP", gpp_sampled_pixels)]:
    ev = events_all[events_all["dataset"] == dataset_name] if len(events_all) > 0 else pd.DataFrame()
    plot_event_detection_map(ev, valid_df, dataset_name)
